# MERT Final-Layer Fine-Tuning Test

This notebook performs the next diagnostic experiment after the frozen MERT
benchmark.

The previous best frozen result was **36% track-level accuracy**. Here, the
CNN feature encoder and transformer layers 1-11 remain frozen. Only MERT's
final transformer layer and a small external classifier are trained.

The data and evaluation stay unchanged:

- five genuine Saraga Carnatic ragas;
- five original recordings per raga;
- four 30-second clips per recording;
- five track-wise folds;
- three training, one validation and one unseen test recording per raga.

The notebook also measures train-validation overfitting, final-layer weight
drift, embedding drift, and embedding clusters before and after fine-tuning.


## 1. Kaggle Settings

Enable a **P100 or T4 GPU** and turn **Internet on**. P100 is preferred.

Expected runtime is roughly **1-2 hours on P100** and can be longer on T4.
Each epoch and completed fold is checkpointed, so rerunning the long cell in
the same Kaggle session resumes completed work.


In [ ]:
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path

import torch

assert Path('/kaggle/working').exists(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), (
    'GPU is disabled. Open Notebook options and select P100 or T4.'
)
try:
    socket.create_connection(('huggingface.co', 443), timeout=10).close()
except OSError as exc:
    raise RuntimeError(
        'Internet is off. Enable Internet in Kaggle Notebook options.'
    ) from exc

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Free disk:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 1), 'GB')


## 2. Find Saraga Carnatic

Attach Kaggle's compact **Saraga Carnatic Music Dataset** if convenient. If it
is not attached, this cell downloads it once. The large Zenodo archive is not
required for this matched diagnostic.


In [ ]:
BASE_OUTPUT_DIR = Path('/kaggle/working/mert_raga_5fold')
OUTPUT_DIR = Path('/kaggle/working/mert_last_layer_finetune')
FALLBACK_DATA = Path('/kaggle/working/saraga_kaggle')
AUDIO_SUFFIXES = {'.mp3', '.wav', '.flac', '.m4a', '.ogg', '.aac'}

def counts(root):
    audio = 0
    metadata = 0
    if not root.exists():
        return audio, metadata
    for current, _, files in os.walk(root, followlinks=True):
        for name in files:
            suffix = Path(name).suffix.lower()
            audio += suffix in AUDIO_SUFFIXES
            metadata += suffix == '.json'
    return audio, metadata

def find_attached_saraga():
    candidates = []
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for child in input_root.iterdir():
        if child.is_dir():
            audio, metadata = counts(child)
            if audio >= 190 and metadata >= 190:
                distance = abs(audio - 197) + abs(metadata - 197)
                candidates.append((distance, child, audio, metadata))
    return min(candidates, default=None, key=lambda row: row[0])

attached = find_attached_saraga()
if attached:
    _, KAGGLE_DATA_ROOT, audio_count, metadata_count = attached
    print('Using attached dataset:', KAGGLE_DATA_ROOT)
else:
    audio_count, metadata_count = counts(FALLBACK_DATA)
    if audio_count < 190 or metadata_count < 190:
        print('Downloading compact Saraga Carnatic dataset...')
        shutil.rmtree(FALLBACK_DATA, ignore_errors=True)
        FALLBACK_DATA.mkdir(parents=True)
        subprocess.run(
            [
                'kaggle', 'datasets', 'download',
                '-d', 'desolationofsmaug/saraga-carnatic-music-dataset',
                '-p', str(FALLBACK_DATA), '--unzip',
            ],
            check=True,
        )
        audio_count, metadata_count = counts(FALLBACK_DATA)
    KAGGLE_DATA_ROOT = FALLBACK_DATA

assert audio_count >= 190, f'Only {audio_count} audio files were found.'
assert metadata_count >= 190, f'Only {metadata_count} metadata files were found.'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Saraga root:', KAGGLE_DATA_ROOT)
print('Audio files:', audio_count)
print('Metadata files:', metadata_count)
print('Output:', OUTPUT_DIR)


## 3. Install Packages And Load The Experiment

The experiment scripts are embedded inside this notebook and verified with
SHA-256. No Git repository is cloned.


In [ ]:
!pip install --no-cache-dir -q "transformers==4.41.0" "librosa>=0.10" soundfile pandas scikit-learn matplotlib seaborn tqdm mirdata joblib nnAudio umap-learn

import base64
import gzip
import hashlib

RUNNER_DIR = Path('/kaggle/working/mert_last_layer_runner')
RUNNER_DIR.mkdir(parents=True, exist_ok=True)

embedded_files = {'10_balanced_benchmark.py': {'sha256': 'd813d1c89126d9a722781211c3f851423783035c3c11e21f0b6567907fccc1c9', 'payload': 'H4sIAOXtKmoC/819a3PbRpbod/0KLOaDQQeEJDvOZjjDqXJsOZO7duKynd2qq7CwIAmKGIEABwD1sML/fs+j32hQtCd7a1MzFgF0n+4+ffq8+vTpP/3b6a5tTudFdZpXN8H2vlvX1fOTMAx/yMqsWuTLOCjz7Dq7ysdttsqDdxcfPgU3bfBqV3a7JqfHeV4t1pusuQ5WdRN8zJrsKktOTj6tizZoF02x7QL4VVRdXi3zJRXKgld1mc1P/yO7uirz4NO3QbOrkuCnLrjO820bdOs8yO+2eVNs8qo7aTdZWQZ5Ve+u1qL+tqn/kS+6cZuX8Keoq6DL2uvgdl0APPi43C2K6irY5FkFf1e7Mqh33XbXtZOTk3EwF6MLsK+BggFfuiYDZNxkZbHM8NVpl7dd0G7Logvm90HdFFdFlZVYbnEN5fPNPF8uoYk2WDX1JsCOnj8Pyuw+b9qgXgXzulsHf37xLtjUy7xsoQp9G98WbR6URZVnDfZ3nuMn6AsP9a7LG2ymyncN/FmUWdsWqyJvgtsC4EGl8j5ou3q7haah4qIstuMyv8nLIKuW3DvxDIXG56f473NAR9cUC2xpky2aOnhzHgeLulrtWsTgJsOveRsH3fjjzxdx0Bblut7lXZcT1NfZTZG34x/qXQkjhrmtG9Hrq7zKm6wDhL4DOljWt1XQ5Nu66aheFuCbss6W2RxmByoB9QSfiy0S2skJ4S1NVzukqDQNio2oWdUdzUF7ciLfNVfbrGlz+Xy1kL/WWbsui7l8/EcLsyl+16381UBv6o18ate7rijl064qFjBDMOuZfAU9XAE5cQcXdSmopJU9fFXvgKibOFjmqwyGtCwWHRfeZh32RhZ8D4/8obvHCZPvX1b3amj/qOdG/+FnU7eqJzAz27Lu4G2yvcdfQdYG27JT34vG7Hi122zvsUi1la+2MHR4gdWWCgF5Nq+bCl+2lUJSC6Na4rjp/Uq+7upmsbYekoqqVpUYGr1DlLYJdkYO8jX8fgtzj4j6lFdt3eCbNheoaq+BwTRVwkshpVUiq76tr4q2KxYf8isgGqRRu84GFveqLpey/CegWqcEE7wsEJ0E8N8rSfHvkODvXhewvLP7mL5liwWsuMV9StTN7yS3SH0f5dJcEKmmTPbik2wnpZV1x2+XtIrSOa8iE9Tq3HzSi898C+s4ve51ZGSPeltsc8Snoj/x7JSCvjb1AhGrSfJtNs/LiwqXAszXxw7Jpll+XGRl3oh5/udyI0vjb/EW1lYLrHmDbE+S966r3+QZLuuLO2RJQCIxvX2Hs3xycvLul9cXbz8G0+CBBhdC7S7984tNOIHf42y8PUUJM745HwMHDRkD4YJlj1m26nbZuC23p4ZcElX2Jx9fvnv/9iL98PLTBbT07Nuzs7OTDy9/fv3Lu/TjxcVrePftM+gLLOKAmEsKXKaNRsH4b4rfJD9nm7zdZot8Qn2glw3UVAVeNlc7lFXv6Uu0zFnyweRP03RZL9J0ZNRMsuUSm6EqUTgeL3lJhIqXTMOWBGm6gKkC0lrAp8W6RvY8vfR8k6/WRbXctTBtRTh7tMXxut7kZpv48rTJbk8FNCXaw2FY9IEmZjy+JnnOsJu67sSUEdmLNn6uq1y/Xefldhq+7LpssQbpIfQB1iEChICLqAORjCQKbG2eN6RClEukM5KF/+fjLz+TmHn3/nmAfKsVrR4cPmsD42XRmONX402FthAeBAIdGmNPoUXk7PkUlBwN7cXBuiSi2zHoOATiSyGYWM/vFuVumQs46ktG0moaZtstaF6eqbicuRPxAdFeIgsARhMIuH9B8bNYE5O4qlDmI5eEToCisN1VC1h7pIDhHCyAio9Bf5tf4U8eP6HCi4BvjwECf4FIlmoSVqBpGDCenyVnB8HMcXTjtvice/vw7GDldZ4tx+tiucwrIKaNH8KL7x6HsWzqLRDdwCDOkiO6UTYDtc/z8fPHq4PkWqz9pHx+dvZ4fVB6Clg/fiyeH+5/ew36q9QSAYCkXVBxQSfsml1+eCWC5FnkY9SC26+urVX5QyCaHARMJSGZAkPIEFQ90yVoYBFqgRNS/uIALIpdPkGNjwQLckEpSro1goGuJJtr4EcRP7TTT9BoDIsQVKC0vqZH7gKxPapXw8qOwlvob44iG/o+DXfdavx9OELVbA1LshTt4H/YtYS6Rt2JRYEYjDOgXyBUhNOiFp61i6KYvsnKNpfj0vZYiopNcRXhsCceGUkjRGX4su1Q4Ff3s4mJuwfNvKTcIzCsN8KjZkshaKMwN6QWoqBnAZMWFTDnMChWYHmAjdA11JdYfUc4KQsgQvUoANMrB52C9WSDF4bAw1Pm4aIP6oVRiJl1CgigT7Ko89qoIFjnUoEGtRfMo8jpriiWCglwORuNzLELJkkNMJMUDfe/9Kulki3adeRrs0K22ZbYiQ5xbKhLRhk2naByvoQyhvpklGELFz6zWsdf9oJ8mGbSFdBo3mwbYA4Rv5o4pELUA09MMkTXIPKnmnhbUTEmpKbX+b1cK20OlJjBim2nURijVjQJR6OEYURyZZikKMzGpF1nwKUj0dooWed3y+IKTP9odDk5fzYTgwDpB/Y5SIplSlIy6sBSn2Bv7V4vAcegA7fUccOyTBSAKPz5zX+8RlYJEKwuhWHyj7qotIRfrLOGvB70AwxvDT1BeYvaUDRSxWFRYMGkaLMSaBkUWRTMYEtbHQEIc9KqIiw8EhKbRwlGU0YrqIIVHTmcC8YY/G7wLzDhPwD3ByN/vQN7bNzkwsxnNYKMA9bnnoCtCBgFnMgGQF3Ly2WboBtAdJ1aQ3+RbsHADb6UJQswW1DRXeSSm+E09KrQtwQ+FVtABWDxMAykRQMIoh0IDLEehYgOJKqu6Er6QUPEH03GC5h+hEZ9c1QJLP4IgDmfjb5CL7kfl1BsNpK9tnojUFjIkdG/IH1soCs04oH27KnkuiO3d1R4sFP0dRhhJYgnB2FFl29UB4/pF1b46l6ZpMH0SwyReGpEP4l4YWpZuFCT1pq16FnR5lTJFmawaJ3yJylUJFJwbRmIkcV6xOQSMXIuaCYyycckI4RtdjqYToOe6ceSjWG0qq6ExX00qBjb1D1iqM5syCdFrxZvIeydeGYEPwxMiPCdsaRqIzGmg9OCZHWpBQP8M5NKRH2LWLucqaHxfBfLmH/hMNExIdtJUKeUbY8SJDb4i6ppu5gS80LTRticao6Neeuae2fEWHLaozN7LDZBZ7tlUaeotHkIS3+0SMvAOrVIvlQNBx/rNiFFkLTENtJffUwGEJewMRj1PuJ/D963WgECFIc0Q5F8NLSVXhWhKNmKUa+UMfSJGk02b/GvORw/iH3vrcYc6Fb5tgsu6A/Zpy2+s/HCisgq/Aj2x5Y3DKRoAsJ6kOPcT4IHqLu3lQdEqEPhQv80CJ3UULYCHiFsKUlx8wMFPJggjkNkUW/vSfsHQ1H6fEGqrnZlCToemOg3uSlLkS+pLkgSMflRhnsQH3ZVB2r9RdPUDWBiwBcDhkmO/rv7YFnnLYEmgIAY1QSix7NEFXfBiXSWLlaLgzRmxw2uXKCB26y81riL0d1T1rdlUV2zoueIG6xKjAxqE5ie+JUlgBPc5k2EGuGyRTxGoGuBWhl6Vovda7lucBoj7NUoOFVgR55hJqibRgIf2Evrs2JRdiWXJ2X+uTiKR5GRaIFna/EoQ9FFglS/kZNGXHjk44hKtHn1pwEx5+FUDO1oseQwyomXV8xBM712KQPp2F8F/Y5FtctPPGx8ARjA/cEcqVlYdj0AiPLeS8SHJAB7coQLAKRTg04A7+CoYNGmSHhCr6dX7W61Ku4kdSPshzDZbJ8jjmE13dDfVZmRozjZfJuFNuMc+dDiDvVYFOV3wLjSFnVAWOpfjha33UFEcBNTB4/4tldFgE+kFrUKH/q19ogyQptVfzYsx/VQL89m2C9j7KSWuWOBYo7qVIISd2PpYNQf+SHtas0JRwku6pRnOwpDZwkeEu9+0W6J9WyV80KTbSdZm4KVCU2NBsTv4zLelu+oOugXMMi2Lm9yL/ghAv06qa4YqWJo9vwb8t0UnojSg+LS5nXhzzVbvdAQ7kqc4o4Eq6TbrMD9irzJhQkE/88bU3YmgU12QhCTNiDYv9AugydCDrwSNkDwbtfCv2Jb9QnzhRpQ4ILsb9Cgm7/o2kCaE1rOJ7rukNIjlB3kRukC98L9Go9WdqBLQtfh4rjxJyiEN24mwZlhZ+mXeyVG0/9hbYEXF3SMJL2S8A6LdfmlqHUM4/WxUcTFpYWIWfDNNDi3+WqpG0JDkDWXYXAOFg2IYia5oFRg13Xd5qna6hYarFLnJl7Fldet8phOcIbFPrXtGjW+2K5RARehCTNcqBHxiUM8vvYFNV019W6bLyePlQbQRqhGRK4Lq09IkD0fHxEA+14E3UT2GFC8XM5Ge0OrvcViNvsoPO5DKHHJTHQ2YjdCpfpiz6wY4qVRYya5PbwTfAto5KpA5xtI3oi4spgI7r/5BpsSQKVJjF0Eeo9klb9N3WmcWa2wjguq2LTMNvNlRo6fSRCNEQj+vjyfgZ1dsHyUa0fobRx0RSiX8C4nipJmkg0jKFl0FPzVoLUTm+C1JhbRjMWB7EU70rPHvfENHjraG8lYDuILZMAvVXkfPFjd3gdMJusMJH3WYURd2wnjUuN2L5DtCoIVmAPL3SIPjP1mpLjeFnK/4oUka6K2loVcj3BB9HyqtwKNUEYwkMn5i9neFALMC0CaTkUgVfKB/kTGbgBjijfPcTEZa8KhPYkd04Shb2oe+dkzLTQrWncxpocbviRSxW5WVwlvbihgDtIt0ca1BUO8baCZdNvkK6DOdScCeyJjfCnISrG9x6FDA9thIpyHN7Iklm2hGB/FZxXKHuNzojgGLbZHFHZ2JNm/KBeVCHIzWZXL4xiJy6K9xsmjqLoEn9Jdm13lkcaW3IS9RztSRf2Y+3xS4lvYMraYACIwilRoWUJKQXlcb9QVo2xWljTHXErAChkfkTHIZFO3uKe52dSVpYv2d/FoI83ctLO23Xh2VOEH9mhqNuTjQrKWZEN7EyJOoB6jrcPTx4lHsQ+7usvK0Gy4r16HxlYm4UNOM68tY56lM19wUhvUvmfsU7fM8RERytGp4nsLy1tGHG/fDwzUQOPToU3QP6I3RLhX8143FFob1N8jLJbQq+A0OM//HAfPHNwArdJeqVEe3wwWXzV5bhfHN77ie7mzKvf0OdxArzOoFIrIxxB/Ky7GOmMsl6Dg6Wwuhb9VYfBNEE5DQPB334/MT+8/XLx5+9OPf/80UUYIGhkcFn2TNyr+MbSqeUBpXTz8SGtcKOHBg7XkL58Y2vCT2Z4t6diQcGDCOzVshRcrKUeZGZAkbMNfW71LmYuIaeyG4iT70K7QF6Y9bkDy9EkFnBRat1GBElb4cEkAGDszLBnpPXHUAc50/sKwZGSnwLIn3UGKbafVj1LJYpWp1+ggGzrYkl6KUmdxxkomrxA/RpuHV6NuEogfaHe+I/teS6BHWJPhIjD5CQD4ItbhkCmTGgyeOusffYx6ldnmXvWBAvPbuKeZPRDTNQc62rsGt8T8G2QDt3VzjZ4MZAzoc7f5wyR5ttoHP/7gXX64qH/DlcmKjSlVoqNViuONsTAMf9xloCF20LscmMO92L0CYyVraOLxiANPPscmYCnBTDLGqdrGEMXQUUBHM4BDXs4AnzfEi+knntCg348rm31ac0yevwbfult9qPD/J+5kyx0SXgfQ63zZanV+BcxItshHVLo1zM/4NrvnQSSGv45HdUmjUGYcV74cP5v1C9IYeyXPfSUJS7MEj5DoohOCqjGaSu2ZUMuKygOpeFqtdtW8vaG/0A5PbwnzBGRtmzed247uWdEC9WKISxf1yhBCRv8yGEbXYTiM+2OhyAgNGq9cScplOhAG5ET0cPxOLzzHiOwJx6nY0A/hhxnzw2FCol0RyVWvViCJ22i5a0j4TgIR9+mEeqn3JCLIFUP9rLYJRvc3mdg6Esp122UNGm+bAoVOchYHm+wuOsMfsiXgK/D8Qpjyol7OkR1Q1gRkVBq7HYM3uhE/bECXAf6vU6uTvdAKGBGNJ7oEEPbMwaeyqMggc/qn4QsMSTzjAYmsyZmnR19mU/HREKjnWokDc2N+M8SS4TejINVJMK/r8jHfGIbRSXZMZ2SAkOWeheoUKYniY8uKocS4VUdtHKs4Mu7Jib3TaFX5sp1GMUP9HUbZH9Vnuy9OP4Tl2Ww64LuRqjOy5+LYIFuFmsljeP4S4cTeiDZF4UG+L2Q8rPK7a+OpGYkpVsMqK8pdk4tNdFuxSpkzG0JvQL3i0kgBNjVoKP2ix2LN8YDSHrdUkmhvexW+anJY56DKPOgG99wRdy+4t6lN2qHkE1NeOpE4K4f7wqn8SLHXUxJoxg7UzLNnrDbnWOcWHKy/EmE+nseeFdqHKNgy+kIGGHWPQ8duN/pQ2+yGCObMu3nKLdCOCLfl3wMXVlSKSrHAGi24wficHgaHQ3naZuoNHe7FFNRVzQQ0WITHwKcYIn44EGkksTodjHAe3ut21EAaKWqB9jqdDLZNFQCbIF22gEeB3wjEmbPSxwb8kb8PKPcfb4n+Xk5s8LOhQTEKoXdI1Cgd563qBAwTpOT5cJP+/X1FdnIjjbbTje1jIJknUot6Mhvt0/aBiHdy9my5p10xLzwhnzR30gE23vLtKiEfLS31WJK2yTO91RRfv9TsZ3YwLO5waBx70/T+9qM720Mb8I72/UhFufOuvbKPVOBlZBwKOHJ17b9gFTGP6u1cmhulzMWmPlVnsAv9yB0L2l+PBibF57863UQ5ON9a7v5vmWYr3uILuDefrCVtgAIUKCIEsHtUjSb/565oyNt56GTK15DWUfEeR8/tgQjXL5nTr5rPL5rLr57HMEcvheBFgKhDEbr2dJ89WvIrptkXoCu1ejlj9vk09mVbVgo+pGp+hQtbPo8Mz11+U+S3IEPQ3eUcolF+PPIiPqFZRhfxqXxDJ1TMF1qE7Sd9/x2XMTAIxU6ttxJbCJS12972hHRgyrFcTs6NALHjd53DjzXIYXaHSwNggfkFyFzbNvVNscxl/pSqrsYtCtYueH4mzs5yB3shSK/zDnom4paY2XZ10J+O5LeqhyAxG/0dZD3Jls0Ya8ls2e3qrTTKy7pLZZx9u9tssuY+Ot5YiwN9KtPd9+wdLHAd1tpc7hlVPX/0ZDA478HmNbHiDKaDey8MvoY1rO0ywfCuN43QsASOtmWXrIormIgI/uAR5ml0fh4H0nvSVm1C9gRijYKkpgQxDu6mMgp3vcunokdxUDfLvJmKDXgqKjuU7Krin7tcxW9g03ddgU7kRuRpmT5/gUdKATLuNYW6HB2YilROIRnHrE7bT5j+AmAm+qQDd8kEgtvwZXZf77po9LVnZxESUjKeX2WyW26L6fl3Z/rzoqzbXB3npcMjzHvpoDj7/ide6pJaPx2l1H42KqqdbcIxc4u9qJuNQ3MDRDRkuTHNCKmANpjRg1jbWpqXqGYlORLkJGvxzDaaCKQXPn82slahqqUOA1M+j1SfmWaOREdA8UgmCSCWCeuVPtsTf6FvRTjSMHjTdaTRbKRI80c6yXozoMAeSzy8wzrxw9ThLpuCUqqkau/CZSmpEwRwgK9wH6XvTnYYbaPwQSF7n5q+lKTafg6t46AKxqO+PHuHFL00Cw6c1RONW40K4N4JQhalySAm+tRFY9z2qW/TLfCMMneo0gjeQWNSD8agsHAmUbC8DM3XR0DhPVsLgnx1RG2xyWnWlq9Ohq13mwzkcot6FBAbEzSyA5BtED13twxY4iPHN8UCRQXnYeLHKFzslhkdlOfX+IgnCLIbkN/o9o7k0fjFdifmcomcQAEidnD+HZ1FJJgJf54GAjZVN8o+f2btXv5WYfIn8vcJRrAPQI1/YGD7IHqg9vYj0brl88tl2iDoji+bUILHizFIi9yvILJEE+gG3bUYtLWpyaW2dAmO80xNdTaiLwUV85hT6v2U/lWxpIctlsdjCF8p3U2mxMBNIvqrkJgEP1UiMByzNlAww0+4T16xHzDsxaZ3ObDFa1Ddu47y1SELaHJKvPeLTGtH1oMMhteaGx/jhpeaySf5TVZGowQPKNBUjk6GmZxJ4sTwBuiaUuOxnM2ljHWlpxMnbGlgvHUlHc9NVl3l6IszIqEMsTFSLmmDk1peadcfzXWZbU95ZJfc5EQ0/Y0B3+Ysprjv6RQG4FgTPccuIp5QmjtB55gBo6UzKKJ03+JVTcYeb7IBeTrQYuw5vEM8d8B9y1wp7SjJWzsNt2b+J885IxxCyufXaScER3QZmq9B7VT0NeU/cdBbcUpH6qDpjvOtUSwig6QDY6H9MeyF8DuVCz7jaGchGGzIftFbEwf3M2j7jPlnUa3A0qpEvpNo5PdiMbEqAniwMTax8ArGhQgL4xxFuOfZUSmcwv2Qm+5LkOHr1qWL7lkPSV44VidxKwZhRk+fmrBHiVXK5y0yZN0vu+6X1Tvg3c09sdpB/5FRJ99su/uUOJbnHN5j7FtZzz++/zXYUMuwErHVdbaj3A3LXYPyUFvfSpmRqxmQ5BrjCuwHZNlMNWbuquD8L3RquMyxCQ6moSFgmrSyBGML1icGICZ9uAZ/d5eFPSMiLwdJI+Cp1sdR8G/T4Pz55KsRBnxYaQjMSmAg4VDRs+HekUbS799eFA/4+S9DCO4wca1OasuhXC0MzYe5g/jCCB1MLbPN8Rfi59+/+/4PRpAY0xIzM1EW14cDXdh/6aChv95Ru8O2BHc/fYtXvqNhdMlRRinKbKvfs56zjOsWy7tYjhkPy1S7DWWedea63/QWTEKyTLgg66qgv2CG4AiwNz0fJaADwxvKngp/+ybx4fFcqh6q/SRu1EDXcajyHsz/uTaTDQtGIfIpGwrMqa3AOFv5BgSy0QBdi2ub2lDXAq60ADxWiNjFelfh9n0GpuP0bCSikvAdToA1GscvzTVij0OTO8MWGHdEhO0cjvGcSeFfzzH5swtPHSBx4TkxbcfBROyAAvUZ42+3mH7WPeRtmLa2EFI4nrpItwvy+KcmMmJHX8ABTc3R+bCpjruSh9YrBAaN9qOMbncYJ0cb2+bYTo42ss0BS+u2ZGWAz/stEpFwI1JRYoP2rXHA6JCI950B2mTXeASonuecDlbm0bWi++RLTRz2ofcobDF7buhm0+1tDveZfqjzfod9dbufHdkvNl5Nz5OBzR2Cn97m6FqdhvLwZxgP7JzfoYnSTJ+9OBuAR7vezTQs56ur1gPGGbHxOFMZyKhPYD5zBvRXCgNRVSVgqe9KeaIXZygFpRBIJcVA5lUsVF/g5uwSlCJFv8Djg9RALg6rBiLvptgPN8OkdlvKFaKaMNgNNJaAnY1B2chuquRj/s8dKrdgElsjhE9vkUP+DGZYpHrnoAELUfZrXcLser/0jxdvf436r1/zUCIxpMFWNGgLIaPY3apBDAPDvAWaNREMqOOlxOm8aWmYL3p+KhNfkdDidYwlelPJKuATK4xAechCG/6xcCvJ8t6P9TybF8A8itzzuTf5roeYSEA4hynRtoiCfW7jaSCXpUzKrWIr7CzdkeDw5hCs02a9NOMK0EAC8kchUpL/dHWuAMkM476aIOFh7WZX+ZTrAcP6nEP1ZXFTIHMB2W/CRvw873fVll+eZOVG08ZkxcH1lErHUiqSDCcHjol6vQZHdnpJFvEWzOgYmhiiMxUG3v/ESSKbiZ0rnWmp24HsNbxVnl0Hce6Y+mpsDsjXSinyJDijqE7em2vzTmXhArToJSc8EkZ8v344cU6Tcx+Usmpi6hLhzFg7Fprf6MRSXLTnhrvMNXpN8EfZhsBdotLFR5fyeI8JdGSffBpdns3gfzP86wZ1swZrDWgUaw3Q7sZIU0tRpfKWjVSL2eFdLK5y56UU+HDf/wBD8ZXH1/fHMKfHzxTz1m8vxJzzsJpvDdrsC1ZXMeltxTFbnJn5O0ilQa+8q9MYKKEyaHFx4WTFOp2Yd4HPQTML0eRC0LUJt966RgfkUcWpfe9E5CiE5Phno8/suMHsBsrdG+0Dar77VjJIHoG4FIU2LxjCj/IN2p1ZtcMh5rCYewfoGX5Jt2ZAbX2FRmSeltdD1F3V7ucpHa34Lpbnm3Rpk4+3691qJbbj9FvV9an6JUf2B282ccmDyPhi5d6AlmZl2YcoN388SuaJ7Z1Gx4CgVOFFOTeT5itFakrHCjENempoV9qW1It7avw2UvKzymaAEW8E2k2XMplQ267YwCRr2qI3yctltvmvyN7lwggJ4Bsw1NZUGsvGaK1s9Ac2B9JlvsjupxghbBL1AkNuGw6/B43yVVO37UXVQWfv38LPSC9e3gbQ+4h69eil3d8+WhctZRlU8m+OAUKw7q/N/R56Sa6e3lvKYq/i5KEMHu2hzPb0UslULkfnbVHJOI8DjQ1R/pvg3BSshEsiBsMUKGHQGIaEwJMzZ3Ua2pUekLG+HVEvu8Zr+E7uVVH2PJMj+DalMGWR+OXSil3wXhW8HyqoiCshBfCqyUjfSLs6xbPNnq37EkzRTjvsRUfcMi2WUAQUcSU1yn7pZA6inYyP0UnfVQ/Ux5cNUTwadjLF/Dpp5CF6sl3xK5rCQ4Ntu3wb9XtBs/uNOmyC/ULHEZjvwVPird7+9whAHsjkUYN2coUh+cLxuMwx11fkeCB9EIU6JYDJiRyq7xAvb9mefPHOEy5YZ4r1Ch/1itpkrys7g7b6euKsDWk04L6RbT+YeIj7eNbdwYaHwZAaFrv9tWtL+wmPOJwfrHeE8WRIDWJx3kBlTzoPYkZgX9FfTyYPyRlayoMiCfbUI/YHK/esOPu1r6KJXFXNfDlUqWeTmi8H04sY9CFkgVUvtiY7DvguEytvtZYjxiYWPf9Nf3M4qyF68E//oxQ29Lf/WUqo/qxyQBxncHcW76KEznn2HfXha5EjvxKrmlpJKaHAqJfapB+E3ROJwxFMVtHesQ48gGEW+NvUEKHyMpj+Rgyf5jgxp4Xx1E/6398GCT/RzZDidkTjWsRlwQEz0vvNiQT43ijqnzx+zzijCAwDcbobZiktHyXrHA7KFat6JA0z3DAh65giTndzjMBtUc14hukLZawuPHybyGhdrICJ0ihYVwTf8uoH20s8G6t9JnwlU35Jaz+0IWV3N+Sa1sSKR5xL0OvDecmXLuF3sGLABAjHY7z4ptyuM1D7XtiQUP7fcdK68IL61P9+L76TUjjOWSv0dYuUCY4J/iQnyS12fgweFAdycaE+HA/R4mganr4K9GuBKo7nBSov4nSA/kEzd/7IzJ3bM/cRRZz3e7HBQCrQnp7ZX8v8CoWYInsg9S1PrRVWNfEuWrlCH4nyNkK2DY9HLwZWuXOIAUnYybbC6xCGo7zpTAJdvslXI7XCCaK9KI+0JJwTDMMya7HXWr47+aQ09wFRqPmPkxVKmaFQaNgQpaLa8JSZygbtUSpvGKJQwWuWckYscTOZCdSyTrUHWxGrHJKjtBiJto5Fb6KiyOxjIVg8DuRkCe4bG63KK3voBl2fZ5hATDxOgNhwcU1c59iJvcVrO/K8aTWUs6SCgXmsk5Hwn+hWvS4v3eagz+wohd5R5HvWed8yH7n7OYK661WH6ry04/xqvT60w6ECwEJhUTgeVhFG4E2AKE5W0MRSMbFxN3iOxyuedQPDJ2f+bApjLGBydgJwQFyENfLgR4XGMWBNgRG2frCO2ECwlsgguMeLjBeyjZXKY8hzEjxocHujMSlR/s5XH7IGx4PRhXxiRZ8HcgSEcS02UOPnvOJrsS1wrhTCt1dNsYzkQIz3lmQ6QrQcdRqICLlrAcsDLMC3oRTQeAWlD5LtNm9Ac70runuRg+J5TJuPz2DEERpzBgMIxsH5KDg9DZ7Lg16Sbbj+eMfrbsDgRjHzETIDvDpZi6qKgl6ga3gc5plm2rqPU/0zNnymRTcNtwvzgj26a1hFGofZrquNr/JaOSSfae9WObf3gjUNr/KH8A4kDw/qchIHZ7iC7q1X5zN94I7naj/MEb6Pg++Mo3TQftfhuPuH6e5gNd1PoS1xoE4crmunuLQkdX7vLgH6t3ee7nJm0Hz/lSDs+by+Q5dcVi3WsMYjXBUwvBGQYL2YhrstnqYr81UXft0SAJaLLRQVRnaBXu2c6bPXBZpHO9RnWKgM7qIh1gbOT33hltbh3bHBKygN+EdnAlJtH1tD7HICdZpbxLwYiXCUHScipbjaPSut6E/obZMeqgW4P1CH/G9uO8DTDzXDLN+sY6iHIqOirCzepkKV1oLW9uGbjnornFP74M0xmhFoQuE9m1lpKUE2EKPUIVr2xx6yTZBGrGYs8W77/tC5h+yFQAlPn4HRAWCGqlTf9jw/QtpPjFjWk8e9ao7jkuZbOyDdKDK/k025L53aXxb0sXcO6pinmikJveujo8QFrkqjXprq01ihpOe1G3DYYZhodR3O+j47OhRBHydUIfYgfn9iJgeUzchTDno7iZUgztPFrQqNTXJCR7G17RvkfrEHacZ+rVbW9B7QASPXaIstW3Nb9o4JmfmBE8aroxMH14Tuy4yvsvOQulHGsD/NSF+Ll3m7cik+xsy7ZtJ5UAk7zbuujc+0svVYY6MxYX1jJ3FHzGBnQ6PQFcSKNxuS654Bjsyy9dxfli1Nq0ZJYXeURlAGuRFn6UW+MWOOdXdi3ZoTgmYEKfFX/q22FAfjoawhGBvQfcbv/SijdPXHXAZA6cmXjRM+dU/k/g9Ry7mFGy50EDn2PQG9eGndZv+dOdCen2O4H5fkIrGGLTiBShkr44pkblVSUKV4fUQXN0lkUeLRoMbAgJkbvyjX9S7v6Hpm5uf6leDrRsuxEPBWuN4yuwESSOd4spQSFTIc+/XjsPY6MBSVRroJfEehx0OBmYa/z7ni03b2L3ALXYEDgQGIuFMrwtpqOzJCUDjhXRe8qdz/u1H8lWz7HTX9umi3wCGixQY9G/RbTPa0p/mwRe+cfZhmjnRfbLLtNPyBTunZX1jvT808GnZNtOLnWcMXsvuOAGR3hlPdTcViWtkPznTsiRloxAeM+Nhn/fuOHviNCtewsHt0QLz1upe6JGFIPcWehMEy0D/L0+tQrcFpQ2YB8gowbenbbPPuSJ7p4J2wbBwE8eH3OA0A+9QX/aYH25BKB93YLLcsv7UJRjL2QyBEOIKCIU7AG/7bWD8qZy27pXseWxYWhwIzLcVq6lGxhIt8epSO49S6n0q9xDIEJLBDqpBdAwGhXnJ8+BcashSN5SWBqfHb50Cf6p+mSBOeb1Nf8TrDH5suU07ZMDnTh9OMT8qLMo9rQGp0gvFrFDoND305QtTLCobyZL06TocabvYPVaWczloYP6hYWYWOw7kHC7H/o38Gjte2fJ07rG15+iYdczZH7xEKXjBjEHg40ifKrA//IyfLBo6KaNYhN8ocEzDUG3O4HdJ3tlhFLd0xnLjapFFWmKB5uyvpLiSfSSoFYe8GHHP5Usd6Zo3jirDmV9ewXscn3hNvfJ2MuHWLhuR53z/v5rINx+Lx5bPsYPGIK9v7ql2/uOMe6RdQ1/AsOk+ejFE8cAeSSaI9zD+6sSo2gEmkQhFLwh6cRA9nfmQSfWv3/88kDsuA/y1zubcujzPHp7d9jjnY8aUJ0sSCNku7G6juLW+0FWjd0Bz+KVAZ+t5dfPgU3LTBKwALqgc9/iCzAxiKamj+/oQ5BDA5xQ69xBmy2GJOp/PL+6Dd4K0v8x3dl3KdXeVjTBndSwWYmAAv6PYYNlCAQDG4CwBTrorMSncZdOuMHHjZvMXEllRExX9hoiO9dZoMdf9Pfwo+AqfeDnxfhWN5xe4k+G++/ElEVu7/2yn3ga825ULqysq9U+qjlbBTJj+U1dzrIZ3Kr1TGRBMRk2C3xRydDwPXHPmgBPr+EKuauqIiOV/tA3ktAeD52bfB9d8/O5AokOtUo/lUTZu8VUtn3n5CU4O5wU/dLwDA+x6h4QdzcgCDdMJ9V+EFcbXMYcXpaDkaEJq3a/wA3+awCPDiDdpl/gsUZHKvG4vc+RgEDLjJAamYmn9pg+LTtLxDMNH3mYIN7dmnj/EiZ7xxuBNleKvnG6OsDV2GpRAvF7E/Zk0jkwOFfWg443mG98xB33DZdTVdgX2A6D8IbWCgxO8B5UcLfg90hAw8/IDTy0bs70xGKtZaPMtTq70PEifyGbo4fg4PfG+YUZpf+ODwFwMQvxCQrO6Px2P5/8nR/wgI+gimtjUFs6VtK8acPxus4cmgasYz6a90kbCTKUBoXqCzmsQVuokCHHUBiitqYRkZsKZreETsXRIjBm8qRnFp93fmlIdpmhrVLm09YtZPW+EUt7UIuzyJosG83SugQDNB2++WD4tQSS/FMJ5odRozL//uSXizCh+w95dPJEE9mU2S5ysBGj/0DlzrEgfAyU2uHjj7rPQjsAhT3r6JL1/aOVHN0zvxZaB74Ykn58+COaQiGzDbxKtN3nTpn19swtmlpOOZSyaX+pC82BkGVmXC+iogQBpdhvQm+jYmsFrLkadlBhJ0hI47EXnihcrI9bFrdgsE65ZynzWf/Kic80G0BiECq4ySK2HsyAi+vyan+/gHdroHEV2MbRXpwbYZWNjPm/F1fEoYipoJ9IxJvVKHV6l3hRKYyyd6o2KISmn5iNL2doSHEmWQwVdO68c1KOQg8Cgh6SMTugrf4NH7dS7M1kA4VE0dYVNUu5a1B6TRcZnfAAUoOQVT+vTpA5Hn5BscytOnyWNkREo0arJ0lCFbdFo3jkHVow6JmW3XlCd0jrpfjvoJJbfDXO/VIhc6MELjPECg4yElgqZcAXDyESOHLkCXBnx3dCnjT9WygM/MXxdQYLODv7288HgIAw/oB8UGLZwMNG6gl3W9rDEWFOst1jXGvELj2F/iMwI5Iuf57bpYrCllPyqoYK2sKF61k6ml6hXVbDHIqwHVs6HFyKo/XSSJj/JcJGn4qCE+il2ggbfFpuB9lhZMo2XwvoaxYvbU1012OyfH02OEMQ74lnevnq67ayY0RyWyRW8UTKFYjqC+3OP9ftVVzipcZqCBZSgrlY7RIhTEl2YCwbtF3my74Kq4AQNpU0Nrm7yswQakS3xgoYh5r4LvnTox2WSZPjhD/YWO4x4lyhmMXwZYYGzUeE3nZgfzVtbQ6QZLFfWy9XTOOqZj6CTBOhP90xHQ3DV3odGcLuADtgpqsyBlsiKF2ZUEPwEWB8y8gGIkWmORLNb54hrzH3p7O5eGAc0V2wZiwRUt5cNrMMUepuTNOVAEIN8CHee0rFY76Bcsqnzc7ag3t9QsLJAGBqBZArCUZbYB81ecsRdLDP42OWYZgxaYNj3d/KDWAU5gIXZDxVShBFliCqIOA07rCqYeEVJULQiwjQRLGG07TAhZVKtyR5zCPK+J61FSY2/ZJgOCh1KLs88DaS3S928Qox7FQe8uQBlPCy0AMqRH0rjDWjhLYDYWa6BqZ+8Yv6nrZt8zELCN0ODZzSm7MN1y8RktJgLZCoOQ85NT7t1TbUKp62ZlayLhunWlttkT6Rs2y6vk6kbWfvPzrgJsXEu/ckXXRlsOGPsC7w8X73/58CnZmFanXUJsCw1+lzJ88DttG1mmDs3852KLN28l/7fYvoG/kTkKsDVuMUhMFvnpffr64s3bl58uXtP9jqLswMUYPOiJe4wQv6NvHUH6st4KoOK+LyzNeaWTJi+BrKFrXW2QDp6PSLd1W9y5Z6jz0mgNQzk9jWGHZZ5qbuSqrOdR+DQcyMdbrHgBHBiAfyCMT3Fh2HEDEVsJ5oyoXHWYl8D28aF8wj0fdDGm+CBz3VHUdeJPt1Ftk4Pfj8jYQbHSGI6wBqsgEgcOQOJ3OQbryx0YgxintKAjkqfm8NU9amx71BWQPOW8ltw45XdUU8WJgWiilIgioKS4So2XEb8aGTdANHXduYs9pE8hXUKn6to3iaoLGai+uKdI3PKjb2pwCxlbyG5Acq8PcoGzT5hXs7egXOlOvHK/HK/4E5fOzXN1sdmSuflvRt8izR4RDG3dBPDq19cvA7V9NgHzc2hjTR40OW7zTcL/8f2vNlC6EJSOMPHNhGcKMPEBn68TDzE/Pzs+V3/4MsA0z+j3Ftc9ccIu1Bm0eqa922bSfrwk19WsVZb+eivEO6uAII0/fYvKw/vzszPUJuAtp+Ux0vgPXSB/oZUXokWTpAFjxpPEj3FHk0F1SL3uypPXcT08fcovYiRbBRADzQzw+nYLQv41YYLyE9D6sK5lNx3smDo5bGmrIF1koFR2xSJ8NGduOB5zC2MEM6YVCGoY0ispCls0Xlq5BfFKwCU1wrys3e2lxa3cjy7P6l+oknUdKyFMB7J5Phnx4MKzUviSo58viEW+Lsqi0BEvo153zHh0Sr/Cu9upKMjGy0A1OyEAfVzXdOhGj1+99Q1cffyS64N11q5N0eBDgseKClDsP+eRSRSxhj/F2+50V6wQclT7eKlfF9tUXqjhv/LmtfgK86S2MvDmEmuzB7dAHlRj7g04/juLhcEiW4++8pLDo3PPC6LSl8CB/SMbBxOILB6xFDThC62CrPa6HsoUXuIeK3E5JuAnbYB5PRFPy6IFFveSCJwMOkHickoHID5xV+A7NInkZtsT4nBgPAee5YzmDR4SHswT/8jNJgM3nHgnUk6iIAyT0I+awGMmbyVnzjBk8ao9tFksqkuCV2jO8sYnolqylVM9zV53sJyLolUmf+IteATmPFhz2IzgL2rFmgtpmLE50Q98UwqyOQoiySu+Ac+xj0FSZYSfAwX3tkbya0vX5tAeAt5tSJ0Vu5VUUy5utbk3Rc9WDdq0csKLEVoIsAP79Aaw897xHDlfAatoLBmfTKkefpRdIsiTULuAqYJ5D7wo2Xf/SjwEwQO5rsTQuepoL2BIfV1dZ2am544kdHECnZwA2yZflRQVLEIgtBbKJBDbUx4z1lRPY9HWI8oI7ymJ67WEHmJWVNvHHPeG0ocCXYzZ4uLxibnR5cRZ+tRE/1fvbahUgG5Z47bNqRwcGW94iO7LoTn3Yx6+CDN2YojluodHkJXFfEdexS3l3jhxUuj7FqLO1D4JZMZW4yZbJrz6luiNiuqj9r5boXQZVaO1chftv/7GPNl52c3Ddwr01TQzK71YEVxS3OMKyi4dCuKGRqYCgCmRcvUFtVZ8VEvkiFsezMYxJ04XiJNv5DSltR5kK9yzwb7gzj7G9FN7Zkf4/g5iWA9OoxhhYkxz8HSIhns3cBiMRdRVs23SjXWxupuPam/iykQjIkv2+mvkpoU4cZRPRw5YGPPLuwuJsgfZjT0sIrza153vgTutHrkCtqhWNTLPFebmqK2LO11tkJtP6XR5K4JgG+xJ5LVZn1q33veupMHm+GIvisVCRBvlkbaohGjNmAbRgX/lwpqfKlJk0KW/oCs0OJRAXJSMo/dcr2xMqtP3ffD3z7D8jP7uORFAO3hJjXXjycPet1EqLtXBeXv3y+uLtx/7TEVDuVQVZ/r2td49qAdOgVIKwJXnhnHNuz3XjFgiSckVne3X85GFju/yERWXzPtQQ3hBkRFYoUZ9xMidexsrnmwAj2AE24qHjgnFfQitHyPxgdumbRGrfDJy70GKWfEoFD+Wq2n/buh/NSCI3G5GwI8TpP6vxeuY3R4MrHnwW2M0MFCfBybKdzfJxA128tewot5lwIHxcuBa+6dPHwYd6itWlR4w7iAUCRWHC8Mk0ISpBIrDAU3eRIqeyO3jO8oi8o/oqRNL9XVdHQz46FfYewSelXbFJDdMHJUu2pvh9SZKJ1AoxJzyy/yODzaaJkTPcHC2xIQZYei7at1aG3kVZ4mJjEsxMMSIPInPz1q1+D8X27DnfXQF7RSdw1a+TBNcn6MooCPf5uLw3qe962mZfb9Vb4qqaNe4j217eT8QxkAbtHD2ROHsyb5Xgdm/3LJDN5fuoF368HSjRVFdReZU4gZvgffz4AjSlO4LTlPcpEpT4bnlHauT/weYmrFzhMYAAA=='}, '11_crossval_benchmark.py': {'sha256': 'e525b1b436ffe44eff8ebb105c902663349c814cf9c874e763a85cf14322bbae', 'payload': 'H4sIAOXtKmoC/9V97XLbSJLgfz0FBh0bBt0kLFrt3h7uci48tjzTsXZ3n+2+izsNAwGRRREjEGADoC21Rg+1r7BPtvlR3yiQkqd376ZjxiKAqqysrKyszKysrK9+92zfNs8ui+qZqD5Fu9tuU1dnJ3Ecvyk+icm6LlfjqGvy5fXkc9GK6N35+4/RpzZ6tS+7fSPo8VJUy802b66jdd1EH/Imv8qjV3lT5V2xTE9OPm6KNoL/dRsRbfOiisTNTjTFVlRdlK870dCXdpuXZdQW1VUpJu2uLLpoV5R1l0bfd1ADcejaE7G9FKsVFGqjuloKavH5i+gyL3N4XEVtvW/gdSOWdUPFxgi8ipq6yzvRRuKTaG7N55Nu09T7q02UV9G+agWUhFJdhP1Oo48b2eFL6P9lXUH9RlAP1k39q6j+BXAob6M8KotK5M3JrqkvBYBawSvuDuAtgA5lVIl9A3+WZd62xbqAPueNQMJCzVWK9D45AaDbKMvWe6RslkXFdlc3QKKqQuSLumpPTtS75mqXN61Qz1dL9WuTt5uyuFSP/AdepPuuKNXbv7Z1pX5v826jftet+tVAL+qtemo3du1fi926KAXju6zLUiwJO4Xwq3pfQbf5+w7AQ/Pq20/YGn3obndCV3lXr/al+Aiv9EcYHfX1ZXWre/7X+tLqHvxs6ja3+rIDlsHu7m7xV5S30a7s1Pdqv93d4rtqp17toKPwAsutdHdFflk3Fb5sK00SYK1qhR2n92v1uqub5cZ5SCuqWlXcl/a6BOaoUmaSbFuvRKl69ra+KlqYJe/FVSOAM2qvzlZ0TbHUZEpOIvjvVV2t91j2XQ5fb14XMFvy2zF9y5dLYLTlbdYChwt+pyZHFvq4VMCyLUHjt6v8UyHa7LLelzBN7PLrqf3UFuWm3ouuE/bbrt5l173WRm7XdsVOIEk0Z8hnr1QjYFYtkTaGH97ml6I8r5ZAyWYcfehwCJvVh2VeKq7jcUCmbdNV3uWq5mv4/bbOqd5HUbV1g29a0clqv6y2qij+lm9hMrQgaLai0SPxct/Vb0SOM/WcZVMNIPHtOxzfk5OTdz++Pn/7IZpHd0STGGp32e9fbOMZ/J7kk90zFC2TT9PJ71+8i5lw8ZLFql226vb5pC13zyyRK6vcn3x4+e6nt+fZ+5cfz6Gl59+cnp6evH/5w+sf32Ufzs9fw7tvnp+8/f6H85fvs1fZ/3r59udzRCk5TU+n4+g0hX+m6Sn8Ay9GJ38+f/k6e/XjD2++/xOVIpzu4rIBPKZicjaO4lVT7+p9By+g8v34SIkzt8SZmHxzGEagBMIYAT1XYh2VMHTZJYwXziIQGMkomvzBEh4zgoUiB9BHUZOAOIUZm2Wj9HPRbbIq34oknp5mek7oxQskRjyi+sU6AolLYFJxA/OzTUYMGf8DkQ3r4HsQcbCCnTdN3ST6G02Q+L34ZV80sBhtRAkLXdQum2LX4Qq4LYiPZ9EdAr9Po38TYie/tzAItHZMp8C+VwKWrSaNNWhGrd2JJXTNleopvs2QVbmzZb2k5SKJ+72Mx9Qv3VECCJj9AKtbBGspPqclzRD1+mDX1/ErlBFEMKzm9ll1VBKWB62PP7/nHiACCf5jOizxgbGAbsqB5z9cphEwLyoJHTjljy8/4GzoM4vkIlo5M1hCW+YftZimPwB3tLt8qfkIXjYASRd42VztUW35ib4kK8HdBFrPs2xVL4HPrJppvlphM1QliSeTFcsaGANAI4fpPI9b0paypdSW4qP1J5t6K2wI+PJZk39+pkDBgth+ysvDoK7zK9SzCGJT1zZOOOYH68LM3O27yapobDxUuxl/bg+3DyvxBNFtAQRqAvOi6gywFwfrkjraToDNCMQXQBA3y3K/Eqp6TvrLPM53O1GtrE5dLA6CacUV/mRUCKsgLt88BAj8haV4pemxBva1YJyRgD4A5jLvlptJW/wqgjhMD1beiHw12RSrlahgWLdBCM9ffHschtjVy014TL85PV4dhEUBkirchcPNt9fFbrKqP1c47a0xbWFhFlnX7MVhhoT1fSkmy7LYtV9c2xgnh0BIcSUh2cJIyifUzrMVqKoJCs8ZLWTjCKbWXsxQFyahZSQzrVMABlBJt9cwKxN+aOcfodFxREtYVl/TI6OAayHXq4Hjk/gz4CtQnwLc5/G+W0++i0eow25gQSqtFQBRSwk1QmcsC4yjogLeASZBOC2aL3m7LIr5m7xsheoXqprFFeiEtDhhl2cB2Uu9WxXL7qLtUKWqbhczm253GpnY2JGoKK3BYM3QcMtoKrIgzADPYkUtZp+mUs2iykoYEw6sJcKjVYBNSdLXETxLzKyoQLjFuHRSNfkWa2coRyMBHQYFr2jwld0eSLyMJZ5sUb+wCrFoy6Bb9EkV9V6PbRKQJFtp0KDSdmJF1E3lR/o2snsm5RYBZLklG+p/6VfLlKRy66jXdoV8uyux+Q4paOmqVhkcsPZ4P9kehTbECgpbKq5VhiwrhMWqt/VFml7LjLgWi3g6sVUWBVHGrIoFbZ2YS91Ldl7DZBHNrgEhlXD5mce3xMrwpObpLWlIczOLWllxTMOWXYtbNWlbAdMiB9HRzpN4DNMznsWjUUqTFDUZa0JIiz9tNznI6EQ2M0o34mZVXIm2S0YXs+nzhUT7cl/AFCGys+LaCjTgxcrGvgSRcWEe4Z/FYgF6OHbI+xYsKqcszcYlugNQHwNJU4mbLik60J1UqykPSTKC/5RSalX71+gFqqV5dZtg/ab+3I6i383tIugCwvcgg6I+1IPaa/wKZcTEyAigKenu7KwC/hURihX2gLXUlPQgAXumINB1N1vo4cWCbfCyzNCLlBUrkHBEH6SKKYBgqAMoNW8Qb2DvK5GYTllosysMzMiYvEXAkhcLYAfU8ORPbIt+3+tK2EI2VmjbhAHibx17hgx2wpaQmVuYOWVQtVNFEgv7r6PpKPona0BcawgQoXJjHCPERIDcEygSEsbOQ0WygGxpbqHWL6fJc8E0WKSswSGbjIKl7aGxCl+w4IW38aJfUZQ2QpoOB/HB4TmMDi4Uh7tEw+0C0eULYrg7tGfByPK6oKYEcRbMjHt6gWXHZqZgI4obDOPkLSglHYI3CBTtqmj/WqOco/fUt9GjqvDwhOswqY7VwLUeGTgjF6rA3sNyrcXIyGF9t5NKFrjUhjFlutFCG6AZSBm/0f54BU3iNzAXojszR+7BFm27KNdSw+BaGj4kZOUQ64blOBshFhjpwCxXXbZ7xcXkEiYpIF21iT0vSL6qD3eqrVk0NY3BcyT9+ArL+yOS9pzEpnTPE5Bou0eiQOfzhvADiUvOdxhXAA36c3lLTv7UVZyJaMqUb3CdBG0T8Cfd/dFLGvlAsWYGmrPUtSUMR62ZRWyR2d8sNWkWoa2ipPsSJuVlXZfhFRNVA7lAbvOqWCPdpc9KYxI9i2LdqRQVhliNmVNHO6jIdYReGG5ejwXrJ+v4Z3KgLvPlRqy4GZAbDqh7izElqUlRQW2iTdxWG1SSOlzMe2aD5GBaxnGsES/drYA/jfcW0mbbNUIkuuTIHZmHWjascrZZBYqiQF0Le9+g5z7xBjR6amukUvlQvZwFx8xawfOi3DdCLvoncmbSI862R89Koxeo17+stqTtoP3bLuegqQjQUGAIz06lq4CJE1uk7JpbV0KtpLGFqzpybyL3S9Ir0WXqI1mZcxIs+X5V1DTCIHrdJater2FeYg/RvaaVfvk66QlGBX3c+zJoL6j/tvlN0p9hMF5n48DMG7kAXKzb/BNxwWlPKWHEkdSyC33ZTtQYRxnqrpJuOBeS4KLdo984vLY386Ad5PS/rmrm8OBnxpcdRAk/jMIl1RjMj5J8FNLBUOGmHo1ADXcnVlhxocJArWqX7oBOkn7J6diflhML9ugxqpFqgf5ezFywi1AnmEyAEXIV/MkvW90wdAs3QsJNAaVgvu3FSe9rQFav4zueFvla8CYD8sMTtVg+WYzus/aOuHF2+nx1n37OP8X9mbFOPzegktGMHCv+s6VUYK6wwFJKRLArd8G3ZO0Sr85wNUykYG/r8hNYmAMsZbwUqH/MfH3kQCXpz7BUrgOFma0tT8MDuP3+AVzN8uDreTQN8YqUFvPQ+h5s8hKE8vVJEMrvHgZFLSRq/O6ePgUKgVHH3LXGlQvHB2He29rwUuy66Jz+oIDPW3znwn8gbJiesUAlTfIBwBndW6u4hOI6Adk/6agr+JDpNkllGevaowfvn8Uf6q02uJd6c2nX1J+KFQZ87JuIDHaegGkUe9tvH4SI7mzMnvQxe3Lf21ozvXJUnbGeYu5mk3wp9VDpQRC2EnpYlxh/iaI65DOVeqbxCWvNXNJx7rkcYTENOdxOPEsL+hLNe8CeDvkKT6yFQ1PNs6NQ8Ty8oXguC0d3brV7HvBxRIwb3Tmt3GtnjFEYjBWDZm7PbMKXuP6746tI0attmYnhmjj1wSLV7RsfFJLgboBm90eo8XMlftlT5BAOBm6tSnvjzoCQirsml9WH0CgfH0DVD3IcsyizTG3aJ7ZKeB3tofH4LmIt2UMCEPtm8bpB1nfU+5AvOqjjh0bQYFhUa1QwYCnGX3LspUZ84jio1nXK+gc6s7DndjAGub2ghETUngP8yhXVx4QiS7bvKyP4ojtC7Qmi9mRxD9TyMLqP/vzrWL7lJu8j/pvGnsYsbemy7jK5DZI8XDAdk3OR2cTyhBR5C9vHTzTqBVTbrVKM3nmDj4ZcrrKj1A5ELeEGPfWBV0P0n/LnC1TfFtpjhs3zB09fkXs1WBElkSIYVx8drs+qiuwNuv/zG+LmXdml7f4SR6JNpuPo+Rg/46buPJnCwzfpC8nEbdWml3mDJRMcszkRZRzdzGO5qX07Vxgi+Dm2cHEKw7Gsy7qZx199890/f/fyu/gR4PTmqIQ2taC9/vbFmxdvJDTZVooqXFd0pUjiH5viqsBASBOZqWe6XWvq1PJsXb8G0jgn/znWNTMqv0m7AkQ9bqFsQd2/Kdp5fBOPOQoUDaIztRkNBO+Kq02Xlfltve+S0ZduqdLQgZYG4yXV99WumE+/tRpalnVrolBkSCu5rszGcaICZUSJ20GkkTHfbNYZe3v1m4coGOTw8V1bFCuQIVsddFsZgGA3YYhdk6u9WA32oeThCspyUrXZctK9vUdipNXuVxZQoPJRVOTxaqqkqQsS2jT5ZU4yMyggXA0w201G++hAHHILmCJj3GeoP2c7YMJSMBVQQ2fAnuT3d7XVfzT6XOMCfi/SZb27TfomDU4C+I6zIImdCIS4xDhJ+iUlgVv73nHU2dSw6J7uq7KorhMZumZGleNExadiibKYAy75MYmX+1VOe+T8Gh/Tos3yT6CD55cUt8f75MvdXiK1woAPDYjMvem3CIJhpvx5HknYVN0qe/b8xB5IjPHEobyTs+Y+AhvpjkHd87DdUYtqNB3XmVARnYBPKNAzpUi1HQwdx28nshXccNsD5zdiW6MtAAxqTQE9sSVUChTtgXJGSMF1N+uCbXhlkDIZ9XBO/5rPo1SAFpGM0q5OmCBKezxiUh4PfPRCATXx0+i8wmGP/o2iJaLvcbWvREfTsRHNvkpRbVzeWzZZRKG38JIZDcQzKIMkuITaVKV/LNG0iP7GcYxz+sMLW5c3ZmvzVHG8y+J9p7QzsZ3CNJHVG3cqgxDsihyHV32/sCdkcAa7+CHzmrrLGnU50Ck81TMJtQpUKLaoZ565bttGF2g3+U7AsoyFpkdKTamUhZxf3Mb7D67BZ8bw79FxNeE0MSMMbkWrwB6Oe9/+J9XstUC6gfqIqBKD8Yj2A7V6urD65TAc+vUvJIUu6MuCNUT6bTbwfVKPFr0l5r1o91sSTWb9kqdg7iyi3j/z7Fvp7TAhB3IM9GaBNU8RFQvW2B2fsbX+W7owbTM4y2pkryYn3ogyDNo/Buoo4BeM00zi9rXVlCHFZ1ST6marNxMoUpd951Q+sYCPjTRm4wYXIbRvrCmBIVnsqJAlXcbS7Y29vQEL2nyglbE314gcAYHLy3jW0ZkC0DZ3nRX75cl/jiFr/WghZ5+X2JjKmZglrnbhvl4YOT7nP+PIkvu+scPBuRnHemYtHYoC0G6H7l1DF1vFHaMkzjvoYMeHRdrreIQh4iju3TBxv6MXfr0FibrwJ39ZCu5r0WTm1R/sW9FgzCgF6iWBaBKnr8iriFry9KmN4ih1Svl+VkuN+XHf/bh+B4tvc0syLOh5tcqL7a67zUiP80T/Q+Rh/Keffo621BqwMba0yWH1F6s0+hkq0zyJcHZF03+J9Hoh1QxqtKWDZo3Yt1DJE3ZmjbXG2yWXdTQAhYjzccQryVEpD0JFq2GrghUEqfZOz2R7EwIJbXwCzc4OkUB3oiOKe6cSwtKa5XOG4tFBeuEEilC9sSzhRie5XT3QnFwQBndgGBCrqaB2bUVeJbBWz6ejFNRfeEOn0uBv3uKMxU0qqdKOBtYmM9DAzUWV9IWtJ/Idcpra/4RHTkCnPsXhtYDO3dpDWg4gCi0vr/tdRq0MzPYlkK5CWi43++q6Rc8BWOKn7Bzhd0hzh5qBXSGudWibFzEBIf9rhp1A3UAExsHRG/qtmMVuLrvYL6NpNMedRFQ6E/3q4D60Wv1fbcTyeodRTujfNkstWpgK0MDSH5stmSMTIhR+80NtdTD6DCJTrZYoFhhZXhucZcjWX2cDA/7bDbY/0HbMsrRlCQmm/MVRd+FCLYX15V/F0t43NM47H9yRTYIwSF4wQ1xoOwWePmUKj04ebmXbzPOBNhYPuCWkSV6y/Kenq2UqT+gmepdi0Ci3IkkOLWBSdnNnpDdrm1+LjM4/JzLKWu7ZkidJne10AvnVS8NNFw43JHGLJzpj/4Rnb286sGyac9Zxfxb3D92Gt81fzWVXwvvN1Eb2WaD7cK6P2sUDMR35Deqzzfz5i9PTgdAQ3Hhv5nF5udYKtyNRPAFjHhcnth9fnhlOpN2M02ZmefDGkqcEap94bDv0sb7ML4uy6AoR+Iy7iCogkv2HnruQBn7w1IY6F6x39d2DwgmjPLZRtEc87h1n1oAGDjofhbjNl02dracakOvXmA6DASkHY5ZfiTkDAWb9VQCsVfGpQMaan46C0gxPSJ/1O+Apkf1T1H1GlUiFVjszhv3P13NUHM7G9mAG4ioYOq92ZFYOF+91VJ2TyK9gml3h5jgL3eQhPMYHHVbZEP8yKPc9cWG3h1XU8grZzuuAI/uqqfc73Ku00TFxfuozo9F/r7fXrch+3nzGuD8+ioPbaTLU3ZKwaO2gu1MCkL/05yBeSsV0Xl4gpAWrlXLJHfXAMP6qvtpvc4nMgEBBrdsOl7AtyMXp6OJ0gY6MHkRGXEHkSDx7whtO1VpDsE8WFyGXtbwYu3gfLsOYBNdlFInErdG5TMTxZ5GvkqpK+cy4HA/k0gwswaLLMtzIW4/ZPIVpvGUJp2wI88IRgJE8sK7WPCu6FVCEFSvV4A0lsaG0Et3nukFOAKQ+iF/2aAznpTvP4dNbVJt+qJttojHz5h8WomNOpoSNdr/0n87f/pz0X7/mriSyS4OtGNCaGCPb5aEpC3PiMyzekrDstphJDYNzMPDEtV7M/EBkm1Z89LBVyx05zjM8tZUoyQDPN0GRAR9u+x/wRMdN+PXtQ1Y/y8mg+YMX38aJGneZRMo5OjBr1VFHYK1XeOjNXmhZxNkc7atIvb16XpFRefVfyVWajtel2FKC/4yUQhv+wGMFOvEeaOR9e7SCaYHB/T8JintOvYHJ4WuA1oBSGbSEuXC6hmmmM3YkkhsGjWscZB+CqU2cMVhXxkpARSeVSOJ1knZ42Ma3MbYmy0C5W6thGP1vvxlpsUaSWFSCjgjqXbM/qTfoZggOjszsMLcSoRh0e0dgjT+BtIXpt+xWkOVsFard7Nfr0t+J0hjO9S+7A7/d5qHaVnOkvBx6vaNhS0NH79Hiu7cnVu/AhIbuGxLTm/TlKt/+74RaTSnIAOz2pgVhClN+XsLkYqsgW4llfjvHGGO5C47RvQ0H5IMkpdOH51UHrd++hZ+J4Ul2JJv9UMMbhmMddKUUwtPmt0YfucQwRmDna3tPjF6yv81/S+JIb5RBGUrXcap1G/6u9zumYynA6BTgzHVzpzQA1oJX4nnsdr9FiOmp5d3FcUKl2uBt3uPBefc94sGceaO2Mmj3m3nbdVb1ys3VK4t2+mvI9+ywQUrK/VWTk1aXdXVWAfm8PV7u6lXRGWezbNMv02IJzRMJV9LY9EunmIeM1tJRwN0MDMUJjyjcFZEE7JptFuTTaXo61Me2E7uk3ziN3Nf6IAmig84RjPp7SmIhiLYZ3BRzoYGqyL0Ee+IKI/KnI5j3oB1uEs8bGgKDvKDAqCEbqu3xIu95nzx6AwFnmzeYZnqOekVRtdWTtq3XHfbRwMD1t+/4PXEh0DxQwBSdSLOfjuxzgP39IpqLYEzS3360GlAQhw2j4dR4PnMEerDKMXPdDM3YGm3fR4OnPI+CIo1rrMkQhHHYVnfsdQ/cY2x1z143G2JSwgZP9ko5yweKHVwX40i/1ESAl5wvxfHOG4ltbb3Q8x/MN0/IWUIe//Q/KrFOf/uf1VrQD0LiiDNSuP2ZtiwBuYF4JA6GoWrkPKUpSK1kqIBCbXnw+GT4AIe7+ISPBXGZ3nEOjGCmL3+YG336JHx448SmOxPCInxvNI4ccpX6B2Wz4dmAW/1q00u5+FNbbeHdb4s4BhO7lCXAsIDrpTeDDNPCPFgOp+DU0b1DC98r3Heu6cJTpzC9HxAVcrZcWEw4iaaLC1+0LBw/usnnJcqxVMrHauaNqX/S9qNcHY47wYqfnDnq4NgyKGa+uaTUrr0IuZTMi5mrt1I2i/7qOpKqrGlxyLhAM3bIunjQEuUtTT1lsa8o9s74umvVwXWKiS5j1SyTO0S5Q87n/2IT+vH2srSC4zh+L8B+pKPoy03diopYDIMHqc9fW5lCyNWHSUJk4LfUjSnWO8V8rv+wdvVhi/oI2z7eKD5uDivv5SFr+P+VZfsAy/u/0eblwfnHM3l1rAbblWQZKNtydNSw/MewCD1rz7UM/3+y+uz1V+odahnWu73tNR4gbSiPVKKTID3wGOPg5s2BEwe9fURKl0IwQHuvErULcHEkF87CZRuCYrLiUN5vWze9tw9H6VzJf8feql73eu5pPPJine6wcv/5pzs5UzNyk5e8ObxF2dvEQ1GnXPYj5xSSfwZJnz76bhz9sywaTj+dyPTRMIb0LLdu5qqdlM4WWUd05vmNkafLbb6bx3+kiEbz9gZP8bSZObXzwqqBB48u84ZTDNoiNr+xjhDRv4fP+jz24A6xAoexgAm9zZtbStQxdPzNO8szfBwueKyNEp4Z/xyeiKS8Qhx4PLe22ah6Snnk5HlMN0qHMu3xqZBbDIvwbIJx5FunbmhOuwR7F2NfL/iMD//LxOc/i1DAa0q7dJe3id8+UCZvOT5ZZYnUddZF06rhkSoJE9rpcIAgujXZiAUzv7pKeu4a1fsMd0/nSY8i+Dr2vBBOvbZbBarB21At7e2wWtP0DrRmYd8IZGmilkMX5E7KqSnMVP09nhP8bmQK0OF+mCqJtStJ5LzQY9H/0iOPXQoqNfNwUez7AnAqyypPTq3eYPZpDG+prQm+zHeE89n4xIk4mMeWok2gozcqeafplytRHtgnZxDsUnE7iQ+hYUbKQ+OGyibxn51AVov/sNCtLPQOWo3IPRXllKCU8wtyFkxbTqHwcqLxZwyTr+Hg+ycoiEI4rRRbTLcC6/xz87YUV+irMi+ummKV5OVuk4Oi9vy/QDqSXd7t0fnx3yIbQ+KB5aERPyXJPJVZfjHyXXRmslEoxcHJR693xSfgP5ZgGi6diN1vq3bODbIonrttHZvB36Yv/LO+QFGwz3eJ7OmYbwWR5y3X224ep2drbJ0W0v9T/qn64x6bB+tmfop/YcGdHuEv4XivQjOwz/Rv8ZYG9HFRktc+v8togvi3ZTJUSvfoMGMlevDMLNpzs7BWOTaZO2cPS2l6LAWHVKMwx4h/4JYz09pvB7McW1G6c8LfPUNmKZ76uwyNXVi6tf4m41wXVlAVqg32DRpgsoKFL1XHkR19hR5jLmO5rexylkUpE7yq8vJtZmdlczLBKjDcIc7PW9drDitf78sySexaXqAajWReqaAf3wnBt+38dvBY22OBlqlEb0xTS9R5X0gtAWbYl531VrmLWEayU7NwEsm52WHH9Ns9mkBcO+ofX/5L5czntZeGEjdpZXA5Q7CPMpNJRye5HNPOugCqHXlbs3JvBf5Q6khX/FpsRIFlJjmop1l6hSgb6OEinBp00Qs20kohs5O9ue2c9zB+BuscoD6yN+ulipNRwFjNSxs9C4ZdCiKjHw89CpeluWfwuJAnUnySKZL3oVibhgxQ2n7DQJnEfUihPUUnT3c8sxkzXI4VH6msDJR5Bd8Phlb/FruFj9w1/K13DweitsOJynry5WDmYLXPOJyH0Nt8PFxwyORT/00wZR7eEpaW9dX0VPPzAL0nAyMfTDFoT9eB/U7+iuc8qut4MZCHzZn1Os8ZVZkRtPsTP/ew17Q6xugeAJZnsh1EpH1hHceS8mFu75Sp0q/skjSJeXAo+ted5dHfnBlqBBcmTskeIlmskgNyxcFgMdai1X1vxezwyizjGmz4Uszw9tsBCcbCugcR1mZPnlOaTd2aX54Ej7oObe4ecnDSmN+6rGeB9GIpBst5Ea+W1zxgmvN9dHwnhrtyGFqN+0dj22AeE+yknTdaclHKjcTDKx5p7IH1jq8XUJoEP7m6hH2/wShwhjewAWtZBvNe+G34SN3QstarojSLh8MKKQw6fLf/OjicWjRgDjGpmOl9k8ApQCQjWZSL4a/a0jzUDO9wHCqh4hf6Zex73r627xN4isc6v3bH/sDZxKFl/yFL/rHlnm91O0ww67a3B9Du6VPDfuMD4SO+Xj64og6upg9aSR+0ik6o1IFO+UsslUdqjWAczw6eLFWLKEmAgSUUvw0voLYAGVD+rIV0QMkiWQFFWGaEy7AgwSyjVtRFr5QUM1BMCZxwOeYVzTTDyp9klngWDXFOn3vCGYjN0cqgMqGGoKdKaJtPxWFpnYJHxkJycWEH8yz8pX1DwQv8W2/i9wMyhoWmt/4HzaxDZQbF53HRaXVYjt0iJBKCxQbmjktY95srHqenp6eOlAyt5dJtIJWdoeCiwIgE3od4/JiSdACnoLpk4euX/1J1yQJ5UF2yyj1KXRpwfwSTJ3zhkmQEiMVK6mVo/dGSJMB6Q0vMgxXAEB88RgMkOslAnIAa6AYDoBP1KA1NqCF0OSzwh8M+g8UDIZ9mchyP+yTKBsZFn2PDkGbfUzOZLoJjr2SPulLsqDYXWxyL+QAO6YeWqvIYMWU0oQfKwAepVs7SNrCaBKo1GOGW1ZWK2axWmQOpn9vo/su5V20pMCd4SZGca3CUp/S3lARa2BwjpS540DMVu8Lr0QPqVv87WInq92N+jykTQ5DcYNws7wj2AahhVSsUCNwrOYiE5/Eb7NAxXdtAs3x+B6EdUu/VzYHyirWALyJQx2F6VTWwLh9fW1DhtHBnJXsxtqSriUj0E308PGb0YMIPfSM2xllVeZUY/80oxWv5KB24/mrUFfn1kffvqVB5GWAJpgyYMyaaiTIAY6SbToSn/TbkATfIhQ7OaKWFyhpUQ2XVlZGYatvoT1JFHtvNjq1W3W0lu9EhQBZOYwulACDFihTaRoqXvIxLBX2poDdK9x9MuuD637T3b2zfDeZgbTfmvFJtZgfbsvsz3JbdsZCu6rTo98NgOKiv9gl3Mqit9nsfwihEivDHg4j1iXwMMbsrF6S5qA1eua2qeaDfkWPl3e1JWirVhVYX/lLqhFXayzd7rU0MIPvQVcIJD3o414Qbg+5HqHvR6JZJ5TS6UPeHYba8xj0t4xSkvIpd4rxz7sotyk29Fx1dYcv+ffNKbhUxsprFndwyq/wT2DfZJabGpds7GYb7+jAcuXcRCkG1zPZx32dvv/Q35s0XLwoE45EmtHUM6sekXvNvuVHCAUdWXRPv0FMG9YXwSpQq1NNddeVFUT2wZ7aw/K16plZrsvcf2zV3qT/SQzdm9GRw82/spQuFJh9LcQIJayXOEUyJFEbIDtM6GXKcfjE2bHndAhYmCL2HB+V/JWS61k4FNjivvRC9dnC087JUF8awGLpzAN0/dqgRvx76tvEv+XB8yEAinDPFsdKel6K3nynLWKV0fUUvtsa7RlvJYGUWyUcniZ5hNWUOhcc6ttgAXbGDTBEPeHTQzRv+0ruXG90QfHuUXGiNpo1vVfr6/le+/ca5Dd2ajj3vht+GpZmFrvbol/QuRLdblguNVUe+cXNhYWghhsgm6sgchZpxKhbrrBqfbnPyqFHUGkczqQRIDEPFLHHmIeeiVI6awLIypnE0dl4iIqtVvQalREXW0f1zwDm7upFBEvY5hOMBcI+80EqyqJ8KuXfiAQeclBFD8a+iNyjP6a7dd+fvP0af2ugVAIOZTI/v8aajP8Kc2WDksTXbY/v3V19F/xOISPbgQJHXtQq3tcEDiTAutOKjCW3EKonAI8MdahzdJq+kFVMyfrF1m71fuY7yqN2iyHqVAwN3xZLuOZmoPIZLto5w+S3F9n8c6M070W3q1UCBdTyRF1Hpm8DuI74QDFPx34Vui7oH++7A3S0h4IFbruQFLvXaLaJv101P1/eRurWJ0xKDAWpaDLbjY8oLujouMovOeLtjHE3tPWvs6ZRvGmYFP9AXaIIvLO73HXd49lUrgB8IBsZZOtcVu2Bo6LFJl3m2OSrOxFZuec6uNeFoj/azEDteyqZnkVTJCRjq3XjIbgz/TrGD1KtTrwtOQK8EJh1LWO/ZaXpGFUs7ijeaisnZszMx+WaIzT6oI6/EOrMojr6OMLg6xfu7zT1hw0z6I4ZSQad+ZD2M5vB7uXgN1PpbRPdlRH+LXuncnvQAZrXyB6lnFa+snrt6NzmDh4803FZpfmEV5xeyvNP8ZDJR/58N/yOrmGSARmeSso6CD6WrUzrDZ07IBpUlPYGudPFSosoVE4j91tbI/Yyo3koIxRUvyLWMtV37sInrD5LuCsaU7p+5cJfPhZ+fhSxlp7y7ii564TbtYMrsNQy3fREBPBFF6BeicfFEjeKTxSw9W+OHOABGFlYuPlNYfnDzcB6BxVZ0oGX1pd/MITgDbfcuZzPkkqlvLmIcUJhGNG/AQKb9STpCpHaMv4z1TNbc7CFcyK7NsVbjDjOhLu0aXMMsyOqNs+jbURcussBuxivsnPlkjnQVZJcX3SfUlFBJoeTqrro2ehQLTxwWlgwso7KXjti6w3Z4+P/j3/EakBU9PIITKD0kqE6r6M+OwRX9eNmK5hOrGX8fg7Cx0Luz76QXDXPIVXRkNOyYyl5LLvBXDwE8cq3aw8h7Dag9HpW7J7x744VRf0lXecsEWYDVe1LV3bOb4bCjQ5s3i2Beni+ZE256qoNoGqzcXZrjbYeaczj9CD3WMAm+sudbIH92HHiHs1RrM0q/kufy+GjQHW1c28w/uj8KiJfmVzoxCgNZPhiAexIrcdQzk7pBwbV4exD0uS10Sa4x3xkhlHdeKpMZrWuSNVkgPRh24NwkgFMMNATMH6GFv1m8xVgrpWW0FzE+Z79/saVNV16E+nqHvTKwU5g1cRuQfPVl8MKM6jIpCun3gi+l+7gRUtv1+uv3fx3bRsO2qPYt2xOE0YQuS5HMJg0GPZpgpPj3Yj99eqd6PiFKzr7GcXj6ND2CRvxxA+CW/ladWjNaWEgawdfSfa6bbnPLxi8mxwHMSlTU8VKDUngoxZR8AQzmZb5vBVk5z1/YRiZYg0Dzy32HV/2RtQWv0+h7ahQeoIIym33I3X6F1yeS0UnmJKXqwUOkxpDji3HKHDm+ttZJWP1rh8QecBVWhfURLNlyTI0HkFK4HudomWO0DOcc0hnDrFOARNy2a2qgoEVZDyfbP59GL0GWNVcSnj1mV/kOaSfwKnUwVdEer8EWg/aRtGPaevUg67LQNGMpvSGYluQSt2eLLV7OjncS/VEOJZZSn5lOFRCy8UHvd3hnOm4JIoPhBclNgV4BYAUYgfxTjXuZS8zJg/liNsUSF5IrIXEFvsAb3pZ0EtYHna/yHWFbNFHA20LOlXYnluhbiXabJm/xXuQjg4eKVrEtFBxUt36AwQQVXOyO1Z2QuyrkQiH6MDd3dR2txWe+2Da6bPAWRbxapeRdhT5MmJlNd6sdJ1fQRBvBCKz3Jc0exA7phBTKdf45bnaFvF3vkOAByD/x5gH6s7Fa3RYUNwsoVkAxvrhRTSRsvGCabPNb5d3A3RC8mbsN4i0oexpOG+BbvMQKaIisia6w1Yq5HuN6ULsGDMtb2Z5YqSOul3ipW28i4H1i211HFwfKm4o0o65hkkzYi23jtLD3RTAIm52gSLwk/ksl3Rok5Udycw4TYcT7bj3BC5TlwXdoJb8SSqdJ5E1r+swvX4SCzlTpRl1uYLTU3bamdPQs4nXNXpNeOCFSvxa7uAfk4BUyBfLQSjjGlNvk+/Offnz/Md3arkO3hNwlGfyuFsjB77Qt4nhIKMcd9AZvdkz/b7F7A38Tu1NgsXzGk3aqyPc/Za/P37x9+fH8NV3EKcu6Zqy6s0d2eubHy+N3DIZBkKEL4yRQZgO6lHzMdWixQNS62hpeTIWW4QS58bPWitJqDa9JDjS2prs3u411YQS3dVXWl0n8NB6Nwmfe8BZTHPYDHQl3iOlKdR/aIbmhYI+MzgSFmbhcZz36ZvE0Ku4PZPig8kpZOe+s6OxgTrze935qvF4RzJNAmX82MGmStrstxTz+vIE+Y8IL6/6skPMY77J7cThY6aO3SsHTSrTFVQUyiZJjSyewyejxTN6SEY+cpj3nd/Sv0dnp4abxsj//MnSVgJCErUHLagyl/gPiv45dRBi/VBf34m2E0OtG/LIvMKWzutc3j36antItch+/QR0U7AjOTG8u+I19Xd4SDnOSiwnRxmZDkDZ4J5JkH3l6bS5/gJ6LSw7Vsu8YL1Z8UPFKNHwunovbRZq67noylz7F8EuBOVF+0FYiaVV+xpuJbezmbAgWtNI3+DkiekgoGSutHRKowYJK2HrpJfrlWOhqnw+8FkvKEY9uPQuVsd2cvYdtMYqu/JBb352ry4Bx0Jw0vAhaXMaJ9ehiyORUJyPQV+UhEe2B1NedFStV9q9tXWW0+W7TCmmup0PGw59iUVhJ7p4+leeFkNgadjzTw37vyYlrYv0Mt3loSGdOVknFS4aB/fI+G5OkqvfNUhh/FEVByIoozOWXxACxE3/rGjLgM5M1ZfSdV8k9ykQfNzUl0zE467chZPXHh4y6l3FTYbotGnyT4iU4oJIVvwrTMCbmNI3MQddPDD5OLCtpsSRAr4tdtqo/V5jh0V33JMRUfU16OKXSHrL7GKKqRU5C0UZ4eCjvYr7IGIeRQhVExYmKKMFXl1PXeh/v3eg7jcqmrmEN1fea+fGVbiiGu7HrvfeWO++ruCFVyfo0MvlxMEXmvgD9k57M/p41Xf0twdh4mQmk0tmLSvex72FWMz+K7mg7Fya8DlEEEkkYjpiwNoU/0EBg4GAFFZ1huXhijcmTxX1Ej2PLdFjHfg13rLCSejNmtHjceI9EGyWxHRs0JJy0J1lqB1I0uXQdrK0Dkmg0VGVO6CK9ZPKuSsoIInC6Yv5finOxeEc1Z2VjU2udxxyeuhL+am32ewXoynVu3WYtNQslXgppQweecceoQXuSsq4ihnsVPUWXyXmbGOCmQTf8K5ZF4bFl75MK6JPkdTI03d2HtlPkdcfI7u9+fH3+9kOf2a0AN11xYe4wpxEzZZLA5dpOIB7tcXCr3mkd1c3wSUR3uPWYWTcJ9z/ygBrkemd6TNKjMHkoZAKVVnOoq0cf5Zp1iRPI+3WEKDRjvT2achU4vOof4jSa0cDpp2A2iAE+daMAlR0v2bXfVUnH4zIgDMlJitfLSvX37QyT3miFIbiBBp4RaqMwuFN694AT5f1xdYMk+arXmYVnuPTTp3fDmX9YmNzR9pE8/ThcWEVq6BOPcmPLYNCLmBg8/DhwAPI4wrwz8Vti7O11fBnKao/PD4pcHDt2b9pxEz1KeE746gJTl2fL9pMnBg7MO7t+CjXjcWQloD15IBZ2ZOwXIGFV/3IcvG1uRAJFwMOxcPyJLD3cxOBNAYv6PGanq38dcUE+2flzXwSaPJwO1rYYcAoq4g2JN1WsRyd5RYYdrDrozmTpbqkVjpSUPi3yVw05cB1jNv5L9UYfJLC8Qfr+9tTRUK2eojpRXSV2N1yT9z31BLRXpy9PdF+e+DbyexU+eye7QbeAnxR4nSmqAlmGN8vGWYY+oiyLVfpwyp7/n/0HOlP7uQAA'}, '13_last_layer_finetune.py': {'sha256': '691b6ed03b90e52dad3d07041448e45c96a3a499408f7ded424c8a0250492cfa', 'payload': 'H4sIAOXtKmoC/+19YZPbuJHod/0KhqmUKa9Ej8b27kZ3yquJd7yXOtvrZ3uTqjc3xeJIkIYZilRIasbj2fnv190AiAYIUrJzqffhva3EI5JAo9FoNBrdjcbvf/dsX1fPrrLimShug919c10Wz0dhGL7OCjFt9oUIyiK/D96ef/j0pA7WWZHmQVOlRb0uq62ogjy9h3/LImiuRfD8ZFqLZVmsgitRLK+3aXUTj0af4Iv4vBNVthVFE1RiX4uaytfpVgDMWxFcpXlaLMUqqNJNWk+C05dQbllWq6zYwOPs5CRY5tmuHqUAnGoAEsub6V1WA4QyX9VBKmGuq/KLKBgCAbb/6t27QFat6iYQubgVxajTjzqom/RegYiDX7Dn3c7OTglUGtTbNM+ha42ogCyjZZ7WdbbOoEhaiWC/W6WNWMVIzdEIYG6DJFnvm30lkiTItruyagBQUTZpk5VFPRrpd9Vml1a10M+bpf4l/+TZVbxvsly//XtdFvr3Nm2u9W9AfFVu9VMD1Ne/v2S7dZYLidSyzHOxJBQ0Vq/KfQGdkt93ABKa1N/eYwv0obnfwejo92fF/YhhsctLQnR3j79wcHZ5o78X++3uHt8VO/1qB8imNIi7lX5Xi/SqrAp8WQN99FtAboXo0/t127+yWl5bD3FBVYtColvf5CKtingFfAWl6gy73Pbq1ZldapsWGfKVLvDp47tzp4RoqmzZ0iwaBfDfq7JY72sA/DaFr59/yuodMM2EvqXL5R6Y9j6pgbGFfKf5PvF9XGpgyZagyber9DYTdXJV7nOYHbz8esaf6iy/LveiaQR/25S75KbT2tju2q4Su6pcCmBnM8Bv0iuRnxfLcqU5Q1IZWbGOgddTXfIn+P2mTKHchH7XolEV/rHa6kL4W701M6yl5tm+KV+LFCfL+Wec6tDWhN6+hfbz0Wj04ezdT7+8TT6en/8ULIIXp6O3v/x0/iZ5d/b2HJ7D7TSd7p6h2JrezqZ/fPk2HH08e/v+zXny4ewTljh9cXJyAnBWYh3kgGyyrMq6vk3zZFuu9rmIxnMiGLI/FEe2j2D+At8lyTi+y5rrpADxFYWzmalqpM7uPhxT/WwdwBQnMLH4nNVNrSHjf1WKAuwDTDeYnudVVVbROvwg/rHPKhCG1yIHsRlkdbDNaCzmwQMCelSw651YAm62WIjxbYKUldjm5ZIETBR20QwnhFiLKQGE5t6VKPgreo5zGkr9+gDur5AtqcdYTfXAQVsSuIu4fC9Rx5Yj/Mf0VCECVIT+qUGSf2SZSgC7FAo6jOyrv0IT/qEd/fnsIzLBq7/G+EuxAUndBMQvDFEw/VMriON3MND1Ll0KzRLwsoLqbYGzarPHxe09fYlWol5W2Q6pvkiSVbkElmE143S1wmaoShROpzfpZpOLKc6haVWWDQxLpXhg8anai/7K7WAAlCuYaNNy3+z2zXSVVeGk/QidS/d5swifyZae3ZXVDbDTM5hyTYJLbvIShR2rggO3CN8o3gnKNS2uKDnhxVXO13pcmINlurwWsQJwHL7fgCqssPgPrMLA24VA/eRgk9ASrDdTUiyAsLBqiUVWNJO2qZeDdUnPqKfAxQTiGyCIz8t8vxK6ekpr7SJMdztRAMlbKBeXg2BqscGfEhXCyovLi2OAqKFr6bGGacJgPD+JTwbBXKXN8npaZ1+EF4fZYOVNla4yRAJXoe0+Jw77hr6IXbm89g/pj4MVkYVBAvpxPz3Q8eXNFUjBaV710G4mpsPsYNTEfiDPxXS479fZaiUKmDtbbye+H669qsodzL6exk/i4Q7ciWxzDfNWLNP7XhAnwzyQozIxrbcg7K5hdveCmR0YjVrktFGhzYDWanqhPf/+a8CtZ/2AZn88GlJWHEbshx9+OA4eLGKHob344/DsBV1rKaa0n2LyqAYVSyQNLDfhEbVpzzVUW63GCghfWtVqWwtQe2EfVt0TC0T4PA+AjWntNaqG3MbE+JnKSPDFLvZ/kEopqO97WO+db6DhyM/L/SqNszpJb9MMWNFoewYCFWFgEtjrKVASf9x2JSvYyUSo28xJQ5wEMEB7McfdkNMNUgCBDEDFeHsDi14kH2pa4ScB6YZJecMWfFQyZb0SlooovAN6C1TAgVyLcN+spz+GY9ziXAMlcqaZIWoxoUboTFSBCVAXpAbIOIRT40Y0rZdZtnid5rXAfpFsAjV7lZWvgD2U8h6pv4pG2PkkyYqsSRKzoAODrs0yTtsnUTfzIIduXayyZXNRN6jCF/eXl6YcIJTBPmOO4wkbwKrSmyX8j/oKlCdh4SnDxgxbj3WjoJnpn3YB1Rp8B1BpTaAi9RImkJajY7uWjQZUtl+MGFFyUQBNsBaNPsBi6rKcEFAk4siMef2NaLJGbBUMUM7VNtXMC5uSc05I8RlVapikHPyFBnFpeoVNQFGLZhcE4LItc5feCtyTTdBIswOtvEobVNnrdVyJdGXGXQO8CJFTQza21C2iaUgi6vlpaH9M87v0vk5OV5IBzUeG6tpq/3eLgO3i5hY0z3bE+k475PDXAk1RywZ2VxJwQIChq8EDdeMJduPJ5SPsWVjLj6EFy0JQkwqYMwO6wtbSRkx/B+q1RbciLaIUprzWlRiHPFi1Q10nnJtBsUsQH4ZzM/Y2g6qhHTu1aJFLshVWpPFrXzhjGFJ9gg9/zbdHR2CUOahyZeUREpKZ13I/nwi9oXfnb6cAUKzzjk+XJcjkFnxV3tV+cXNg4sAmpNbTodOczUMX0MiFGZFL4ht4B1CofYdyxEAgrImFFoxz7WI7WGJRpNNCYPM0cUTSgLQuq3oR7prQN0v8jEM9S2gFqGn0sKMX9mt3qNMG2mqkyam+aWvFIJgi9+PYx4bYklxAJdLdOShpKHn2EAGNCJEw87LY2CW8XI1IyGYMTx9oKVTyMpzztUECkfzvgcCWDN+0eAO71Te4WUUz1KtW7Y+KIn4r7RbHL6iwhU0IKi6WqOubnTptBRKQPc4HpeXPA6kgdpbMPWwko3Hctuwse3qrA1OjtbvFZJvZAb+hXiucZcBY4GzaglYIqk0ltmUjEpRMDqePnRW/SpQ1fhEA3wHXOVykX1poToJQ1QYtCdWuLnsiTGcJwoI9646DC7OK4RJufx3jyjQ7/foFKTzXqxGZKv/48i3MHvSWlFC1uRP5rdCIKBdF3LcSIXeCSpluRYPWusIexbj9ZNkflZ4vv8TK6FQnuEUH6tOybHOFscC0ipAmwsV0dnkAG1P7m/BBthkZFSVbkWm2XfbavqLhPNvEamagmcLhbeaqAV2wiD9CQyja0tweIvhEE/gdSPuI2nO4CgvAREgr+XXCZmO35M/nb36Nuq9/kvM0UvO1twUDesLFwZjzbytPQF1PaJLi8rOFUTJaqW3FJXpQyT4BEAtYLNyPbCS9lQ2F2+8tbsAbd2m16hN1fIHSi8knWkxMGXspsksFv8mZunBmt1pMxfZK0Ho7D67KMtds3pGPN3e4XZ07igMUf/CurOrxkYsQG0sUIWgYt+lvmrpwF1hszH7lCH2tteiRip4+laDGcnz0BABxaabxDjotcDbJr1IVBabimmhebjKjFJmxjGRlS052yOrRYBTAiWp85P2q9tXELrgnl3ijiV/k877FtFe36+hEoWFZGDNbV0JPzlzu3eMVrC7La1gYl7s9/psjYcdWcRRuWEVt91G8EZZ8VhDyCWIGQFDBrhmQR8OVoaHuvwwtNhmPQeuRu8Xc4dAenIExUf5HLO5On8nIEUEdulGjDEv6ecEH73LMarKu9dRk9L3Utpt0s6nEBktKfVH2aleVV+lVlsN+2WeS6DNDSBD2e+pls4ft4wV7Hfh/K46V/pDEwgKm4MUl/9oaIezXEgfzGlmB3tHSW1YN2cGaSJYbMzFHkmkRaADqV/vZh1QsXReR9fICAV2yje3JeOxCkcjr6irKIFK7VKo/jrclDPSy3G7LIpqNL04u4X8dQBJbDYieLKujWVqAysAHy5vI1xG2dDKl30J2sIhEQ28Dyqu/gyo31v4oyWkqSiAa5KADjNfV/R2pRwq+4iJQzvEDhXMsAmfcqs02/WzZHbpisjUvq41DZAcLqOGa8IbGjEphJ6ShBWQJq57IBy/4kWe3F27TZVUm65kfvI6F8MGbBLCBr9KNWEggsGH4IgDWKrvNMNxiceJvEWMnnh/oli++oqv2K6Q6763h6n6+WWxBk3rep/7Z0BfIr1VabETUX7zTUS37Uefbf5W8l975OY/+kFEsFSwzFRkx242vimQRt7Dl1sqbfJqMfKpa7epqvcu+XBWYxtqUTZoneVnjdDiJT1RkDYgefO5OPyNDXVHrylhjTVYvDLrmHdnwZQezYi0q9DtKfZzJYJTW5FJFaa3oaBtYmYYJsKmsa8mJm7LLaJyolmYBCuVVXi5v/HYnW69w1Ni2fVddtVE+Xv31tmG/8HZuqINHdLLb0QYEpGgYhZVJi2irWupCtWGwAU/3TbmEKdM3LAktG/J3jL+HjV8kbWbfdwsJ0s9WHBIu5CG6sBx7x7hLebmWoc8Ep46fypzT/LS2h8tfxp3Vi840PzRAPVuOeu5tz950QA9lV71lDSytVXiLmQ2UHI5WH6eQRvgLww3kj0D2Kp/H2Aun+xb4TAx1ox99JdtaORvpfit2dnjcyMPvFmoFwyfaDETjcfCU7FveulJsQrXeEl49sYOx5Oe6XDeojkh0NT1Bm6K9qE3XAelkK5UKK7u2QwGpPGLcLlRQM13ZjS/HjtwlEa+Bt3JXmopV4XYC8b0mUJQs4S2xnwXY19lEEpGrFRbNpP15WRZLWHwLXID7NFZjb3cqdBRXYxRnKuyA7tq1h3Pw2m+p9YXDc1IS6CI038i04cA1XyVZKaxs0SrQGogWyZMWrE0/+ODuDNEwGUPpRHl3VAPSN2CrXOqlVhq1+wDnl3/PyPvnItIp0OLe+aLZjwez6W9IiESRgQiHL0ad2lYRa9thOuvqnT4SmJ3hITL6kG0pRmi0T3yvocSY2h+lNxiiiiqPUjWPiBzojRro22AttW/SdlXqGGyY1WQlZg6U+nq/XudC6p3qlQ5QUeqnUXSVG0dGSZC/xAmg0J2aBG20gZqlnDAGoBk3BdSMisF2YX6azwrvhfprO5AwllJUsDU3r3cZrNhiW1b3joYkSSaSdbFYWuSSHpgCNlDwSqkmP+tnkLmd0Bt7Q9xa95NVla2VboQeqCzNubWIW3TVsMOeCiNmBkrRuNBCMtc7Z4kX0/1XoihhG+W8NbYzhUvS2tDUC20tM3JN4aNKLvTzBcK5dA12anljVm7ezMJ+PlTZ9Ktdw9Waut9GkY3X1AYNC/zT4JQtiZweHmhW5adPdU0dbI3RSfU/qiZqURqrpc58Yi3A8j4T09mp5oZlWWeFAB7eZnkKCsx9gp7V6Ar968Kax0G6bnB7aV7RYJvH7oij7Qd6IIGBXkMQABAzf9jMABVyPOG0iQt0+MiKbQWA4BTwAFR0MWg8w0pAj2wLqLDmHEKQmTXBqLpI8WO+SsjxzIQSvuTc70hI+OdSi8hjxOih+KpWeh9jkqJNADlMumHzE26qXGWVCpbr38VT76Fgq/TSwzPY0lyL5c2uhBbrEJ7XIZV8MNQKvgtmKlhHAzk23A6D23OQTKtEnfloscCGzUdjSIqL3RfZFvDZNq3ufTXVpxiD8kKtLdlttQdD6GAZh2V/we0z0jimGEziFqZm7Sr0hK7D13hsySXJPNjTYZ624cCQMg7HI2v7imwO61BkYwmMnuflXbLLlje59OFT9GGd3sLCaFsMkFNQaXq4EfdzWeICfl6CxocKOYlbeCbrNH6M8agKc575I1oUZUAnpQhHxLGOLHJhbFrSgGofdQIl3XgAiWM4V8haARx6uefxqfzA0Xdsfo6ZLQtVHmbMiuQWXJbgUU2+4039sRW+uIqhmIqxpfO0+oh6jkz81Ug7AGp5FoZ+AltVUhDhP0wHHo/aQEPY5O+sDU+9A+2RxMQaGinKAg2qEcKzAxSo3EQ6HdA9hQ1abqhHZkusfU10FEY35HTi28HZL23BN3H2uK6yoyVb7NO3WszI4kJD55hc+vjGF3bCSKSti6SAKJI7lBodMeuRUjjvHwwTsWNYAk2qfWFKvtgj5oFHirCQBPuDimVQ/GjMZ2Ou7iVWMIkZ6a9we/5TnthHvcpJBye6u2GNHnXjUHA1z6OBUBvZJg9p4R7ynoAWNlfJTqDa7zR7THNjxgsGzT6mMF0OWeEH8/oZKCffz+PT9ePbZw8SOfbKIFOHIxZ30lqgZFDNKzx1d140wAr3b+CnQYzmXdKe/lgQyzgv+d5y14B++IW4RIpXehOfrdLt3wzUC2vaPXRjrQhtlPgdFuGk7Nogw7yiZYJEgIy0SHJHQDxOvrJ15rg+unVTZ6h9Zl6QR3USOqojqczf8NWiXqY5IzCdfki3u/hnYNSP9C3SZmajMdWgQ6A7xx2YvEraT/ErUvHPikKkGAv75gMznOiBNRh/Ajn3eaFMZYSxPOZldpFHCL2tSOt9hWLPLKdBgycDmJlXSUGp5CeNPDdgu71aKTlhkTG0Ml2EWIFbclrm52cZbd9Ex0RmrdlyCrVKmU+VJI4lesS7JtRRFpV6B4VnIxkchKdx7v2qv3FNXQnUINLiRkVKmZckMTtvdSMnumESmSetSmuj/jWaq6naMpLUP22QqDfs2vPMC76i6KFJvNFDE9bCRegU4TbfliE7wSQcQFuKV5UTaLCeLGJX0rNkuJ4qxauqQUazAyuq3jJfHB9lCkixQLdfOWiLBzqlFdHs0po3cFp2atBHq9uKd9zS9N4pWPUD13Bx1ju7IDsWoE9OwEq836KYoDwEspnQqfrAcHh89sBE0iMLxVXTl8qSqwljfuOdqNbJUoW5jFs7k2wIo8jJO88asEQeIskDdWjZ6MZ1+riXwhlQx4iofJmARi4cZ6W0ODiOcfPBdpGb93Z8ycWl89m4UC/sYOC6EbuJ8XILZSFxgiMwK4QnXKIVuqRf+w4JiHq56BlnRe8HOYShJ15CwBbUPYPkOVxAA7WYMcX9/3vp/1/x0vuc6Uc56D0e99ahKz3ulnPdoWjXT1yld8nXeXyZj7it/EwKGn0UP+FH8W2el8sa/SE/8Zh0YIrftpuprzH1R4LzHFrqUpM+/GGgXdxBn3TDE6TowI/odXZFwZBnGA8PGqS646D6ti/oRxK18nPc47GGTY3MdkMuOexEgnbgxM/0F/0RBFrh7y3xNTtMT4DE0I7T/e/SPy1n8ckx0RiaP4DAQ/TTpKacWB7cv27pcpav1muh2fuYWAa+yg1FNHRWPR040DOtjcVCRVt+RSiDtYTqdnoiGjw6JA2CgQgSJVupmeXZ2Hg2N9ZaCxXcldazx/FJd3YwsbyzLDu0n5Vq21wuy93je2pUwzkf4jZ6gg3b2Ft3ODyzm4crsog+6Y73ePC84a2K7wjnjN4XMv7j0lOWJAfDkVeyHf0XJhK3FxCLgR0C1BbzAZL23R6UnMiCAzip0n6kOqB6sWJGFoBhRANJNZAN5X5XY2A4WkbcupaJpL/2rFP70d1Z6aAf4GHG0HIjZTMVHRL19N/BzSnWR8m2mD3CTqlpW8zHalNnblnmSLMjZGca6flP5pu9WvJNJP7pftTbRe/+e9wtr/d09NfVsplhwR8aJ8t8t/imfZ+9H5h3dnzUBZQSiwck8hNbrjy5nMfP14+TTrVbmRlzecOqdkd7sPrrWaeeHv/BesgCrCY+qvKhhwWUbx/2PNEhm+WAqDYil5lS5j3j76lqjCh8knJbvaeSsqDMjdpxoLg2nMytZXKwkjahzLUg8JQxZpO5mRt95TRpzENfSU1u8+DrFBlJ5nIODBqhXfOZhxd2qxijjl6juhip/qK7JlnWt85kYnZILSGhUNhxt4nPw+k8aOr+aSE3AjoVl3MCuDuXB+04Iq3ye4Bc7igjKYVl2LO899Q0CaRKpDejERePUpwdl/LQL2RWmbR60oTQxuYhE6VplycyomnKOMcaB3rvgEPLsNIOpeBCI1f+lbZt18xywLgtEUbR8w2NOXrmUU01NOe+oWP/00Z7luGTjttU6X0COy5+jpw5GFjoMPPkm750vg9nCrU1n/fG1SEzHasEzh3PBy4XTQCa2abAnF6xu0ZQXB6eHfNF6vndt27QaI/31Q5PQLcQWjH8IWB++vEQ4h4Sdouo+Is83dV9ttlgqm23PJzHjummDJlzrwf/sPhWuxPU6dqdjcbYVtc7dWSuR7eSo053all7KF3J1RVbPVPq6ocxahUTp0I/NvYGqR8TGkNNHj6e/dRR0k8Sh1cZoA0W06ThVbr4EOG6ClyySRHFyLNx72meb5psLX2IhN5aPEC/Emg2uxU8Kar27uJ8BSTpr9v7dn4kaubhkd12tyzfyWO81rFOf926WblV4ZVdU028ZJsV+4ZOFOip+Cz4/oSbmZg+yUIMmFJpHcbEoxV2MXzFjyWYMDTfuYYhweskNLI4pRPEb85XdLnQLuge9bDK9x4fCJUIZJJtfoxYDEkRsqsdEJWPOrki6h1f8Bz2DgrDWHViAZ8+ldSVwtUkQ+TheBMtSMc+53a8L/KsuIlUbmsTjKkC6nK1jLMwAaX3T4w2T2U3y1hlsY+47kRBDGK7gyWF0hNH9pFnFk6ofk06UYGPJjyXJmfKNYFIqz37f3m6AH26FRA4nBPgf+zwPzWn7SCyn//6Y/46glof3CckxpP+E/r6YEFe2gJqC8IhA76MWJxHdzg6YeVfP3rytJIKhu6c7TfJQYeO7au89/ttuhtZcSnYnn0sqT1X42VHU3cy6jturs5+MeVISgrVWNLbggp4t45HKQmgYiopntBcFIBnFzJYGtTh87jN9R/xHvEjRFbGHUkhFOBSMK2zDYbbk3t1lzdxvb/CYa+j00nwfIKf6WBMNPsB7wwZG5c7qJSTIGKRgvW4xwsdRUyTDifWOAAXRuZmFPzK6KZa4xnWGrzUoloBSquO0VC5iafqB59Rk+BGiN0q29aO32NsnyZQNXHB9VekMwY/st1+Vf5dtL57Gx/o9qsz6BH8GxU0e2Dni/HypzSEiRk53i03qtqT4ayZfnx37vGc4o0afseZ3bzfPQXqO0jLz7BdWLyc9JxZzppFuFv6vLbK618VbXLGEN3APSVlomG5vV6wcF6Pf2yIVIOxBR7C/fr27L0HIxQRMX77duIVSYGK4hXmlOyhHihrsGmrMZ306f8dqrjLSn2xDh9wBsd5eYeS5TExV5zQaUOPv8e9BMVqe6KF1vhrGrXvX+lr2HdLy/GNo8gCjWa/LUDigH57Xa7wGDsee3HkFpvSTqTBGjVjlJPcxNcZmwfv2IaY7lW2dzGfBCc9PuLw3io26ytGNx/Me86gdh0wXQtdDWoIqH4wjBXKe0+oBvRwQT3uAv68gO50X98vAP3u6+s9CAN5VUPXi7340eMUT/PddQrz5EfPVEo/L3C5uqD1R47opS8AaQOa0oK8pYvghOw3sjClEx6M+nGhywixrMlFpFgXUxlLHnoMj6n8ucnwePPF5TGF7/2FlU0s6uuSLyqGiIAxRZ2GNtCQ/Bx580GoqoOhS6aJGE/N3OrdALUGK+fpZayauLoqP2PgQVosr8sKlIn4BNSLGaymeblchPsd3smTi3WjqImpJuv9TtLcbL5+2TfTck2J8im5J7fAqSOJdI0YGatxDz+VJyG04Q6vReBXm4C609DuHophukjzGvdqgEQkt1yrXbaY/QA9on5k0AlRL0KqGpo6y7ysnQ2RknhcnV7uq1uhz96bk3tODrderWwGZOVaGahoL+IfmVrGjOZtOORLxhvK0aBjoe15b7CyXjunBJ1vgycGGQju4vAJaCNc6cgZuks4smNb98Y1ghoO+huWjHgZdwUcNdaGu05835jNzylA7S9AFNCPx8B3RElLsB9e+pbffwqxPg813mZRN/c5CNvpNBxGGT0EX43w7J+iZJ9z/l/Y8OHIgCGa9SOmh8+sCyGexpkHdZlnK8kPIDXS+hp2KSZgI2S1Z1btM4XekRBwnqc0vxHWnBFLrjY0yFF4TqThtIw3VQY7G9Wx025v7lVVOmg0FfKkETnDPcjrwhp7b5FsG4HgRIFvt6ZWhnUJEgVFGWwvQXnL9WVB1goQkrCnmzK7Ej1I6S4yeYWlvEPlaPE+6go+lFTwCZNxhPI3XYqFu5o2SFzK8HiHt+wYL5daJJwFRq8KfAnQlxD+UykMjzePsNSgdOkheYjsexCjXsPGULpD1xzC8uNh8J82UVhOKrWyuetau6L9OAl+UMuZ/+bHSN3cGKzks9J+F61BxBYVpC4y9yRs9Rbhnyly3LyV+llSqYtDF89fWlk7yuoqrbgLXrMyn8PEo36lY07MKfUW8hzKi4K+TQvpY62VWFKqRXlWm2UbSFvhwrLW6+uPrJebdNc+E+fUjUrEkq0dYBhucBK/kFqo+8lA52EKAD34d6z1/MSb0NizX3+fVugWtaY7qKTAk9tMXt6Y4e23aIkGta9BAwRQ98XJH5yYHmm6B0x1CEF9DVp0WtAJcrEVUuBOWXgnICuvuS2giA8aRVADLjK3PFEBSuuENfhciZ1I8ZrKTh96Qye8lGjla0CCDnoKHQRddN9Qkz7UEbG6yfLcwTzMMSYTOlYhZlhbefxTRlSCvC9gZ32Vi39T9/umq5ULC5ijhKmO22pcJ/Iag962qISbe4a1Wr4v1pUQXxA6lug4yLv89e8LDxf56fOrAU7XFdONynIS6vG+0v19/v0f2vuMFfjY7RgOu0r4QgOqRxNoBa8kkfE6YoS3xcC2q7IBEVCAhlz/mwuMMUexh+1KhRc/llW2ISy7lFoJgZuhDqP0JOMlZNUgZlsYw1u8hidHeZLf00ji/csCr4/OgeWBc+LgLw3yR8gdeJg1LAdZi/dGi2W6r/GSajaQwfIaxXrdRvoGV/ew2O4rZavg3Q4980F1bpve6FGSR+WJXVpfZBxaeY3uMJQkAcKXlRJofGk7nB0Fhpcugjgyv0uJGWzzvJtfu43VIrVybpl/9Fq8r8nn4drZOyuwZjr0eVvbL3UWWn21VeZg6nxGxVZ/5DE89hRa6C4px6nHA07eCImKtLiZfl4M++svtUNbuRbl4gNwPOtQF7dJf7+9pTSefPkl3R1ddIbzfi/NAq9JU6QEDAH5Fj7JpeMDsRJb90P++/e/D/43qAXy6kx/kZ9KnAKYsgXBHXONOywPZBXXk5PPEzSJBTq6Wt8Ke4uVzKXrKK80dSYoYFeIQtaQ4IHm7VkMYh1nLsBYZ00DOP6vgc6qTGrA5KvgI2bI6Cm7DqfBB7zqdR48PJkET+K/w5SP9OQa8+N/UPI1yhCffEOp5hgBsTjKkPbi20lw+iK4+Y8vUkGSVTQETzPyzHPwPNCaOWw0+BkNlEIzqXEZSebH4xyz0rBSICD3uNTCwghLoYCRIDC0HIjP6RKla0kXnnIo70ouMmm1rvVmzsGL4ub6B+dTmzgCh+e1ZAZUh+q+SlNVai7nwKt37/RNX+11N0pXs1m0DmbT2cyG1LY+97D07DTQ4h3P01Q4zuxGAJtxTPKTefDD9z+iMHxw8pw84suXkwDvcplY6TNUyhNVQz09DlDtP+7RiWUCWHpK/hb8Na0yoi7+xCwnv1nfp9Mp/n/+m9Wb34LOPlR7vOSVd78pTNlpi8fAhWEo0lPbOm/Rrf83CkgKKN1EW4fnoOhWIfdx0OYBaWs5+UG6Ff9MZ4txY8b6pjP2dIv/rM4gBtYZRF3Te0KxC+StdMUG6sS2rq0OiHeKn2Og8rQNVNbRz209/aJb8wN52ygpFZZm/ja7rMtkb1Hl+0Aq1wCH4S1UVYbRoAD8kz7EIBdm/YJOIgSvZ/CCyfFP+rCELvxXJjksEF29v4eNff841FAihjj8O5Bimz2GjqKbnd3s9Jud3WZo7aazEsdUWM+OK8r0oSMqcA1JF3/gypd62Zkpnf27ukgM6x/RQ61xPaGyTy4vnjjHTdwKTOF6woJT2ekWpWgdrO07GePUfdB6VLf37opvYjJk2mBjRTB9xA/YReckTj8kGUIFc3X6nAE8QED7jgh/G69K2K4tG71WG7XDgV6Ve327idGTnwanL8ePz05fukBpzyMzWTh7xnknzRMHOB3UauffOR1wBQydfjgoYKgUiGiKRKeDGUommCNTUnZYz1zOYE3+jV4wmaRXGgpeB4mmnW060LwjbObH/KMqmbi6ZKKvgDTcjMeNKwpaZ64rUvi9GdBx6j7gARc6sIVgnsgcJsT07QcTRQ6f7bnU8sdd3zTszj9ep/dYWn8rbaz1t1QZkiq8yqGQZqz/4lCT3vjmtqZro5DD5B7pto/uh46vBdn+g21hUbMP1d/3oEhnqK39VKV3VxQCPBkEh7P3LeAY6O4r+wUxGVPfJCXq7oFFS7AepmErZl+sH/tQUZPGHMoIrkRzh9Jq13vMpHOO0j52cgDvobE7gDE7+VIyp7uJ/fE0rYwgF09Mf1g4UVdue9aHr2/LEOtgWx2eOwsoAskQVI8RXkiKhlLNIJp7ZKwwbnwcI59jHNySFW69r6BoJdcPrMTGWZ7pCT5dW4Y4giUzrcG+UR41IPMlmQ6AhXDHji6562yJonMjaJMfw3CVt4i/A8oUCe4waipQqSv08S78sC+IvRGnfQ1gG8pdqux1DjwTXGEZsw+Q+ReiwkpPXTKdXgNZ4eXP739FmkMvazUUrdZcT4INWqnhpWt3FgXev36LU6gpibJMC2YWB0JXj2SV1Tdoeu1Qe1uCmqutkvUWdI/WnIBXfh7sHlpSlMnrQFFtGWN5BbmzEOP3pcmTLGfhfxWhNLWQQB2r6Daef7fN/768STdCnUHoiS3BH3NlNF1eozRTESC26xO2+Q2Xc8r9qc2w8ZdsF3ag9B47aPMCQTVMRxz/n2z3OuMxPRwM0ybuGOnaun95n/x0/vrN2ScdEUm5khUE++onzFaCwr4vvAW7+uH8/S8fPsXblTNMLkHk6aGhMtpf3J/Gak0YxVmdYFciT6yW6obkgJ4rgwDEpPdL3C5TTclYAO/PSXYg1z77jmbbsV+grRpEMcG3B0+KMYI+sIMQsvVNXl5F4dNwPPZHigFoGsYBGnwFLVp0srwnCRYv8U208XpeOL+2923gXby2mR/1f3nesxYJPkTWwVZ2mAbIkd6moBJc2TTxHE4Nz6VFMA3+M91s4Mf72ckJLlafXpAklW6qat+ejcZUZLgRkYcKEhCtCdmcf7cIXg63RAvTKks3BUhnWBUE3amtgiwkuGdkfXWbqsUGpXci7bk1NsXdvb62/lOInTEBK9MvhbgE6zSrAnPsxG3M2KLosnA3b3v4t7MP7/7y7ue5ypFHBWfkG00xrTU1YhqmXb4W/bE+8T2Qs1xlbMVAWgwDuBZbjEemYKK7a+BcDLYJx06ifuAKFMURdYCzIgiQMm9jKJVI8Wbs1+JGeXBIPnkLatnlQjw2fT+DfUyV1sGVKPbvOuY6t7AswjpFnkyWaVXAFF0yIXpDXJ5gyaQqy0am23Xf2vexIKxaFmwf3VM67URY+GaHKS0+g5q5Eqwof8MuiJE8L0EQtAWfC+x9p46eJwvf5LHOdmOMQ5KtZFZ5WJVBp5FcDo/ycnbYQqIGFLUjMDa3Pmn+MPQ3TEjl+zlRB4AiAiF7ozGyClFcC0Mbt/DQ7J/PPp7r84SJGkD3yieDkDvCLkZjy7vbgr8uS+hIewmqCx9RcdyOHg4Z5ogeJhi3LmKV9f9qn+E2DF8YR5l1iYgsBzsCnFEJKJ10ULu2LnCnaiwQSY+jg4+Xa9yvXiZk6YMJOO8NYKe0avWR3bjU4mZmu3tmlYslvLFD1dDXDNPNHQbQgdpypVF16GFs5az+lcJW7CVE8uc8eGjJpkP1dS28TQp9mhiqpvs2fmzlfoPZrs35QHZW0tg0UPO5kEcbLi29U8Nj4ab8VrhJx2kvL0QbaEQX+/aGpLdH+wK9hwqtTo89N9pANQtMzI8gdeqyWw4ClWvSqqy+JYrkZLtQIfP20VuWIIJ+2wd2JEfwDYCB4zUXujcDdZLyOLko+LfuW/8tGV3adb+3Q+/JU+u7IILP20nPraO2gLB1fEXnslx3rkTHa0b2OcuxElnTou9+ZIw9TQuX98xVnZz/Okfuj2n2h+9//Mbm3JP6prUkz25E1EFnosAbaZ6o5YtxorqZUhur9QFbJlJVfgR9e586/X7JUr3oO47lJ5YoQV91CvKJnZpzR+tClb9kIJxEB26mF9NJX+VuLgQDwKWir34nJ4KpbrK+aHroDAGX9nRFSnvnqieFmsoaoyBdyOfLQxnA2uI8gXpfAjad0KSt1H8axE3J0lbpO6fRyZpiGmlf9iPmZrt0MKR0LwNJJd0kOAOgPDFjPT3vxanNhXMMlH58TIqcgz3TKW/8/WoT4hwFZ4BAJk/OV0BiWTx7IPWlpjwmfY4G6T8vOli5U8XX/BHJcYZROAjgKDQGE+4coEF/1X+iaZmv51taxppHNdxN99Pyj/vpcjClpJXbx57u7ncvoG6CIAPE/eYAeOxGfEd8Xc9qWHqjzmI3jtOCZ3jGy6l04c66NVTYXaR42QNmt1+kS4hileUmT4eVA6qr/ZI8O20ua72B4Em2fNcH9ymHHRJM+m8NO/rW5WT4tuX+JvuwdFRXjshAn7/hyuSePqswEcxjEqoUZ5zgE5VBqj0ir94/OsHc7mH+VhcZOwV1Ak/PzlT7XWTuTp6v88C2VvWhra82uOq1W5nFQltQjMsMxLuMatSgRn4lKiTvI5DGdwsiFWgNI/qaLI+lhKXpai0lunivAUU58FyThK43ZKzgVbWtw6nXNYHISinm2aKkJFDh49nb92/Okw9nn847SbK1OdncTea/njD0xjLqWt6PE0/G9hZ/+eQU0ZGLupB+HkztPXylWieX96E70EITpqtL+y4opKIqPleXs+4rNI49FqiqS3YvULPzxZkIVV3Dee1U0mlUhMCVuTePSihjuhIdpgVlH7yLp60EHndgwanPVD9/DGSvvjfUMvvco1P31uYfezMeP9o3xHnTF4zZN/tc6zcucn4TEYvMY54Rcz4X2nDPtLLzuWN+SoeOng4mNvPunCe9e2Jm1ec2sN7F8ht6aXBVeBEOnR72LjWmPqnaeqlRFFGO+s5xq+N85rY1qmu3VuvZxN7wO5llFCaWAUe6WsmN2hfgwC264X8Vr3uOiLeZFmPLBkxXHiO5AKRCkpbvolmcjm1jsTw6hMGvnB5PWno8ca3LZyosAcPQ6RdZlkeg9iYJppBJErreNknQfZwkoT6gjb7k0X8Dr3T0W0WyAAA='}}
for name, info in embedded_files.items():
    content = gzip.decompress(base64.b64decode(info['payload']))
    assert hashlib.sha256(content).hexdigest() == info['sha256']
    (RUNNER_DIR / name).write_bytes(content)

SCRIPT_PATH = RUNNER_DIR / '13_last_layer_finetune.py'
print('Runner:', SCRIPT_PATH)
print('Embedded files:', sorted(path.name for path in RUNNER_DIR.iterdir()))


## 4. Run Last-Layer Fine-Tuning

This is the long cell. The fixed settings are:

- MERT final-layer learning rate: `1e-5`;
- classifier learning rate: `3e-4`;
- classifier: `768 -> 64 -> 5`;
- dropout: `0.5`;
- weight decay: `0.01`;
- label smoothing: `0.1`;
- batch size: `1`;
- gradient accumulation: `4`;
- maximum 8 epochs, early-stopping patience 2.

The frozen reference values come from the completed previous experiment:
36% track accuracy, 0.319 macro F1, 77.7% training accuracy and 49% validation
accuracy.

If Kaggle stops, rerun this cell in the same session. Finished epochs and folds
are reused.


In [ ]:
command = [
    sys.executable,
    str(SCRIPT_PATH),
    '--kaggle-data-root', str(KAGGLE_DATA_ROOT),
    '--base-output-dir', str(BASE_OUTPUT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--num-ragas', '5',
    '--tracks-per-raga', '5',
    '--exclude-raga', 'ragamalika',
    '--segments-per-track', '4',
    '--segment-seconds', '30',
    '--batch-size', '1',
    '--gradient-accumulation', '4',
    '--epochs', '8',
    '--patience', '2',
    '--backbone-lr', '1e-5',
    '--classifier-lr', '3e-4',
    '--hidden-dim', '64',
    '--dropout', '0.5',
    '--weight-decay', '0.01',
    '--label-smoothing', '0.1',
    '--baseline-track-accuracy', '0.36',
    '--baseline-track-f1', '0.319',
    '--baseline-train-accuracy', '0.777',
    '--baseline-val-accuracy', '0.490',
]

print('Running:', ' '.join(command), flush=True)
recent_output = []
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    recent_output.append(line)
    if len(recent_output) > 300:
        recent_output.pop(0)
return_code = process.wait()

if return_code:
    raise RuntimeError(
        'Fine-tuning stopped. Read the full log above.\n\n'
        'Last output lines:\n' + ''.join(recent_output)
    )
print('\nLast-layer fine-tuning completed.')


## 5. Read The Result

The report compares the fine-tuned result against the frozen 36% baseline,
shows every fold and hyperparameter, and gives the next-step decision.


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

report_path = OUTPUT_DIR / 'REPORT.md'
metrics_dir = OUTPUT_DIR / 'metrics'
figures_dir = OUTPUT_DIR / 'figures'
assert report_path.exists(), f'Report missing: {report_path}'

display(Markdown(report_path.read_text(encoding='utf-8')))
display(pd.read_csv(metrics_dir / 'fold_results.csv'))
display(json.loads((metrics_dir / 'overall_results.json').read_text()))

for name in [
    'finetuning_training_curves.png',
    'finetuned_confusion_matrix.png',
    'embedding_before_after.png',
]:
    path = figures_dir / name
    if path.exists():
        display(Image(filename=str(path)))


## 6. Download The Report

The ZIP contains the report, fold metrics, hyperparameters, confusion matrix,
training curves, embedding comparison and drift measurements. Large audio and
model checkpoints are excluded.


In [ ]:
from IPython.display import FileLink, display

result_zip = OUTPUT_DIR / 'mert_last_layer_finetune_report.zip'
assert result_zip.exists(), f'Results ZIP missing: {result_zip}'
print('Result size:', round(result_zip.stat().st_size / 1e6, 2), 'MB')
display(FileLink(str(result_zip)))
